# Playstyle Feature Engineering

## Decoding Winger Playstyles

The previous notebooks established:

1. the underlying FBref dataset;
2. the canonical winger population;
3. additional behavioural variables from a second FBref-derived source.

This notebook converts those raw statistics into a candidate representation of **attacking winger playstyle**.

The central methodological principle is:

> **Style tells us what a player tends to do. Quality tells us how well those actions work.**

The initial feature matrix is therefore constructed primarily from behavioural tendencies such as:

- where a player operates;
- how he receives possession;
- whether he progresses through carrying or passing;
- how often he attacks defenders;
- how he enters or supplies the penalty area;
- how he crosses;
- how frequently and from where he shoots.

Variables measuring execution or output — such as goals, assists, passing completion, finishing efficiency and take-on success — are deliberately kept outside the initial style-discovery matrix.

The complete conceptual specification is documented separately in:

`docs/feature_taxonomy.md`

This notebook implements and empirically evaluates that specification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "fbref_2017_2024_wingers_enriched.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## 1. Input Validation

The enriched dataset should preserve the canonical analytical grain:

> **player × club × season**

Before constructing any features, the dataset is checked again for:

- expected row count;
- duplicate canonical observations;
- missing identity fields;
- season coverage.

Feature engineering should never silently change the underlying player population.

In [ ]:
print("Rows:", len(df))
print("Columns:", df.shape[1])

duplicate_count = df.duplicated(
    subset=["player_id", "squad", "season"]
).sum()

print("Duplicate player_id × squad × season observations:", duplicate_count)

print("\nMissing identity values:")
display(
    df[
        ["player", "player_id", "squad", "season", "Minutes"]
    ].isna().sum()
)

print("\nSeason distribution:")
display(
    df["season"]
    .value_counts()
    .sort_index()
)

In [ ]:
assert len(df) == 1396
assert duplicate_count == 0

assert df["player"].notna().all()
assert df["player_id"].notna().all()
assert df["squad"].notna().all()
assert df["season"].notna().all()
assert df["Minutes"].notna().all()

print("Input validation passed.")

## 2. Feature-Engineering Conventions

Three broad forms of attacking-style features are used.

### Per-90 volume

Per-90 features describe how frequently a behaviour actually occurs during match time.

For an action \(A\):

$$
A_{90} = \frac{A}{Minutes} \times 90
$$


Example:

`Take-ons Attempted /90`

This answers:

> How much of this behaviour does the player actually perform during matches?

---

### Behavioural shares

Share features describe the orientation of a player's involvement.

Example:

$$
\frac{Progressive\ Receptions}{Total\ Receptions}
$$


This answers:

> When the player receives possession, how often is the reception progressive?

Shares are stored internally as proportions between 0 and 1 rather than percentages.

---

### Opportunity-adjusted rates

Some behaviours are normalized by a conceptually relevant opportunity or involvement measure.

Example:

$$
\frac{TakeOns}{LiveBallTouches} \times 100
$$


This answers:

> How frequently does the player attempt a take-on per 100 live-ball touches?

These are used only when the denominator has a defensible relationship to the action.

---

No ratio is created merely because the required columns are available.

Where a denominator is zero or missing, the engineered feature is left as missing rather than assigning an artificial value.

In [ ]:
def per90(series, minutes):
    """
    Convert a seasonal action total into a per-90 rate.
    """
    return np.where(
        minutes > 0,
        series / minutes * 90,
        np.nan
    )


def safe_ratio(numerator, denominator):
    """
    Calculate a ratio only when the denominator is positive.
    """
    return np.where(
        denominator > 0,
        numerator / denominator,
        np.nan
    )


def per_100(numerator, denominator):
    """
    Calculate an event rate per 100 denominator events.
    """
    return safe_ratio(numerator, denominator) * 100

In [ ]:
identity_cols = [
    "player",
    "player_id",
    "squad",
    "comp",
    "season",
    "age",
    "born",
    "Minutes"
]

feature_df = df[identity_cols].copy()

print(feature_df.shape)
display(feature_df.head())

# 3. Territory and Positional Involvement

The first feature family describes **where a winger tends to interact with possession**.

Raw touch counts are not used directly because they are strongly influenced by team possession and overall involvement.

Instead, territorial shares are constructed.

Three candidate features are evaluated:

### Attacking-third touch share

$$
\frac{Touches_{Att3rd}}
{Touches_{Total}}
$$

Interpretation:

> How advanced is the player's overall involvement?

---

### Box penetration touch share

$$
\frac{Touches_{AttPen}}
{Touches_{Att3rd}}
$$

Interpretation:

> Once involved in advanced areas, how strongly does the player's involvement extend into the penalty area?

---

### Box touch share

$$
\frac{Touches_{AttPen}}
{Touches_{Total}}
$$

Interpretation:

> What proportion of the player's total involvement occurs inside the opposition penalty area?

The third feature is currently classified as **Secondary**, because it may largely reproduce information already contained in the first two features.

In [ ]:
territory_source_cols = [
    "Touches_Touches",
    "Touches_Att 3rd",
    "Touches_Att Pen"
]

missing_cols = [
    col for col in territory_source_cols
    if col not in df.columns
]

print("Missing territory source columns:", missing_cols)

assert len(missing_cols) == 0

In [ ]:
display(
    df[
        [
            "player",
            "squad",
            "season",
            "Minutes"
        ] + territory_source_cols
    ].head(10)
)

In [ ]:
feature_df["attacking_third_touch_share"] = safe_ratio(
    df["Touches_Att 3rd"],
    df["Touches_Touches"]
)

feature_df["box_penetration_touch_share"] = safe_ratio(
    df["Touches_Att Pen"],
    df["Touches_Att 3rd"]
)

feature_df["box_touch_share"] = safe_ratio(
    df["Touches_Att Pen"],
    df["Touches_Touches"]
)

In [ ]:
territory_features = [
    "attacking_third_touch_share",
    "box_penetration_touch_share",
    "box_touch_share"
]

display(
    feature_df[
        [
            "player",
            "squad",
            "season"
        ] + territory_features
    ].head(10)
)

In [ ]:
feature_df[territory_features].describe().T

In [ ]:
print("Missing values:")
display(
    feature_df[territory_features]
    .isna()
    .sum()
    .to_frame("missing")
)

In [ ]:
for col in territory_features:
    invalid = (
        (feature_df[col] < 0) |
        (feature_df[col] > 1)
    )

    print(
        col,
        "invalid values:",
        invalid.sum()
    )

In [ ]:
for col in territory_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest values:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest values:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

### Role Heterogeneity in Territorial Features

Inspection of the extreme observations suggests that the engineered territorial variables behave sensibly, but also reveal an expected source of variation within the winger population.

Some classified wingers also spent substantial time in more advanced forward roles, producing unusually high penalty-area involvement. Others operated more like traditional wide midfielders and therefore recorded substantially deeper territorial profiles.

This is treated as **role heterogeneity rather than measurement error**.

No adjustment is made at this stage because these differences may represent genuine attacking style. Later sensitivity analysis will test whether the final style structure remains stable when compared with a stricter positional subset.

### Territory Feature Status

At this stage, all three territorial variables remain **candidate features**.

Their conceptual status is:

| Feature | Status | Intended interpretation |
|---|---|---|
| `attacking_third_touch_share` | Core | How advanced the player's involvement tends to be |
| `box_penetration_touch_share` | Core | How strongly advanced involvement extends into the penalty area |
| `box_touch_share` | Secondary | Overall proportion of involvement occurring inside the penalty area |

No feature is removed at this point solely because it appears related to another feature.

Redundancy will be evaluated later across the complete candidate feature set.

The immediate objective is only to verify that each engineered variable is mathematically valid and football-interpretable.

# 4. Receiving Behaviour

Territorial involvement describes where a player operates, but it does not fully describe **how possession reaches him**.

Progressive receptions introduce an off-ball component to the playstyle representation.

A player who is frequently found through progressive passes may:

- occupy advanced positions;
- make forward runs;
- position himself between or beyond defensive lines;
- act as an outlet for direct progression.

This can distinguish such players from wingers who receive mainly during general possession circulation.

Two candidate features are constructed:

### Progressive Receptions per 90

$$
PrgR_{90} = \frac{Progressive\ Receptions}{Minutes} \times 90
$$


Interpretation:

> How frequently does the player actually receive progressive passes during matches?

This retains the real match-level volume of progressive receiving, but remains sensitive to team possession and teammate passing ability.

### Progressive Reception Share

$$
Progressive\ Reception\ Share
=
\frac{Progressive\ Receptions}{Total\ Receptions}
$$


Interpretation:

> When the player receives possession, how often is that reception progressive?

This partially controls for general involvement and therefore provides a more role-oriented view of receiving behaviour.

Both are retained initially because volume and propensity describe different aspects of the same behaviour.

In [ ]:
receiving_source_cols = [
    "Receiving_Rec",
    "Receiving_PrgR",
    "Progression_PrgR"
]

missing_cols = [
    col for col in receiving_source_cols
    if col not in df.columns
]

print("Missing receiving source columns:", missing_cols)

assert len(missing_cols) == 0

In [ ]:
prgr_check = pd.DataFrame({
    "Receiving_PrgR": df["Receiving_PrgR"],
    "Progression_PrgR": df["Progression_PrgR"]
})

same_prgr = np.isclose(
    prgr_check["Receiving_PrgR"],
    prgr_check["Progression_PrgR"],
    equal_nan=True
)

print("Rows:", len(prgr_check))
print("Exact/equivalent matches:", same_prgr.sum())
print("Differences:", (~same_prgr).sum())
print("Match rate:", round(same_prgr.mean() * 100, 2), "%")

### Progressive Reception Source Validation

The donor dataset contains progressive receptions under both `Receiving_PrgR` and `Progression_PrgR`.

The two fields were compared directly and found to contain the same underlying statistic.

`Receiving_PrgR` is therefore used as the canonical source because its naming explicitly identifies the statistic as a receiving measure.

The duplicate field is not used to construct an additional feature.

In [ ]:
feature_df["progressive_receptions_p90"] = per90(
    df["Receiving_PrgR"],
    df["Minutes"]
)

feature_df["progressive_reception_share"] = safe_ratio(
    df["Receiving_PrgR"],
    df["Receiving_Rec"]
)

In [ ]:
receiving_features = [
    "progressive_receptions_p90",
    "progressive_reception_share"
]

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season"
        ] + receiving_features
    ].head(10)
)

In [ ]:
display(
    feature_df[receiving_features]
    .describe()
    .T
)

In [ ]:
display(
    feature_df[receiving_features]
    .isna()
    .sum()
    .to_frame("missing")
)

In [ ]:
invalid_share = (
    (feature_df["progressive_reception_share"] < 0) |
    (feature_df["progressive_reception_share"] > 1)
)

print(
    "Invalid progressive reception shares:",
    invalid_share.sum()
)

In [ ]:
print(
    "Rows with zero total receptions:",
    (df["Receiving_Rec"] == 0).sum()
)

display(
    df.loc[
        df["Receiving_Rec"] == 0,
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "Receiving_Rec",
            "Receiving_PrgR"
        ]
    ].head(20)
)

In [ ]:
for col in receiving_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest values:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest values:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

# 5. Carrying and Dribbling

Carrying and dribbling are related but should not be treated as a single behaviour.

This project separates three concepts:

### Carry progression

How frequently the player advances possession while carrying the ball.

### Territorial penetration

How frequently the player personally carries possession into advanced or dangerous areas.

### 1v1 engagement

How frequently the player actively attempts to beat an opponent through a take-on.

This distinction matters because a player may progress possession through open space without frequently attempting take-ons, while another may repeatedly attack defenders from relatively static positions.

The candidate features in this family are:

| Feature | Status | Interpretation |
|---|---|---|
| `progressive_carries_p90` | Core | Match-level carrying progression volume |
| `progressive_carry_share` | Core | How progressive the player's carries tend to be |
| `carries_into_box_p90` | Core | Frequency of carrying possession into the penalty area |
| `carries_into_final_third_p90` | Secondary | Frequency of carrying possession into the final third |
| `take_ons_p90` | Core | Match-level 1v1 attempt volume |
| `take_ons_per_100_live_touches` | Core | 1v1 tendency conditional on live-ball involvement |
| `avg_progressive_carry_distance` | Secondary | Typical distance gained by each progressive carry |

The objective is not to keep all seven features automatically. Several may later prove redundant.

In [ ]:
carrying_source_cols = [
    "Progressive Carries",
    "Carries into Final 3rd",
    "Carries into Penalty Area",
    "Carries_Carries",
    "Carries_PrgDist",
    "Take-ons Attempted",
    "Take-Ons_Att",
    "Touches_Live"
]

missing_cols = [
    col for col in carrying_source_cols
    if col not in df.columns
]

print("Missing carrying/dribbling source columns:", missing_cols)

assert len(missing_cols) == 0

In [ ]:
display(
    df[
        [
            "player",
            "squad",
            "season",
            "Minutes"
        ] + carrying_source_cols
    ].sample(10, random_state=42)
)

In [ ]:
takeon_check = pd.DataFrame({
    "original": pd.to_numeric(
        df["Take-ons Attempted"],
        errors="coerce"
    ),
    "donor": pd.to_numeric(
        df["Take-Ons_Att"],
        errors="coerce"
    )
})

takeon_same = np.isclose(
    takeon_check["original"],
    takeon_check["donor"],
    equal_nan=True
)

print("Rows:", len(takeon_check))
print("Matches:", takeon_same.sum())
print("Differences:", (~takeon_same).sum())
print("Match rate:", round(takeon_same.mean() * 100, 2), "%")

In [ ]:
takeon_differences = df.loc[
    ~takeon_same,
    [
        "player",
        "squad",
        "season",
        "Minutes",
        "Take-ons Attempted",
        "Take-Ons_Att"
    ]
].copy()

print("Differences by season:")

display(
    takeon_differences["season"]
    .value_counts()
    .sort_index()
    .to_frame("differences")
)

display(takeon_differences.head(30))

### Take-On Source Validation

Take-on attempts are available independently in both the canonical and donor datasets.

Because take-on statistics were previously identified as a potential area of historical definition change, the two sources are compared before feature construction.

The original `Take-ons Attempted` field remains the canonical source used for feature engineering.

Differences in the donor version are retained as a diagnostic rather than being automatically reconciled. Season-level stability of take-on features will be examined explicitly later in the notebook.

In [ ]:
feature_df["progressive_carries_p90"] = per90(
    df["Progressive Carries"],
    df["Minutes"]
)

feature_df["progressive_carry_share"] = safe_ratio(
    df["Progressive Carries"],
    df["Carries_Carries"]
)

feature_df["carries_into_box_p90"] = per90(
    df["Carries into Penalty Area"],
    df["Minutes"]
)

feature_df["carries_into_final_third_p90"] = per90(
    df["Carries into Final 3rd"],
    df["Minutes"]
)

feature_df["take_ons_p90"] = per90(
    df["Take-ons Attempted"],
    df["Minutes"]
)

feature_df["take_ons_per_100_live_touches"] = per_100(
    df["Take-ons Attempted"],
    df["Touches_Live"]
)

feature_df["avg_progressive_carry_distance"] = safe_ratio(
    df["Carries_PrgDist"],
    df["Progressive Carries"]
)

In [ ]:
carrying_features = [
    "progressive_carries_p90",
    "progressive_carry_share",
    "carries_into_box_p90",
    "carries_into_final_third_p90",
    "take_ons_p90",
    "take_ons_per_100_live_touches",
    "avg_progressive_carry_distance"
]

In [ ]:
display(
    feature_df[carrying_features]
    .describe()
    .T
)

In [ ]:
invalid_progressive_share = (
    (feature_df["progressive_carry_share"] < 0) |
    (feature_df["progressive_carry_share"] > 1)
)

print(
    "Invalid progressive carry shares:",
    invalid_progressive_share.sum()
)

In [ ]:
print(
    "Rows with zero total carries:",
    (df["Carries_Carries"] == 0).sum()
)

In [ ]:
print(
    "Rows with zero progressive carries:",
    (df["Progressive Carries"] == 0).sum()
)

In [ ]:
feature_df["avg_progressive_carry_distance"].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

In [ ]:
progression_carry_features = [
    "progressive_carries_p90",
    "progressive_carry_share",
    "avg_progressive_carry_distance"
]

for col in progression_carry_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

### Progressive Carry Distance Diagnostic

The initially proposed feature:

`Carries_PrgDist / Progressive Carries`

was intended to approximate average progressive-carry distance.

Extreme-value inspection showed that this interpretation was invalid. Players with very low progressive-carry shares appeared with implausibly large values, including values approaching 90 yards per supposed progressive carry.

The issue arises because the two source statistics use different constructions:

- `Progressive Carrying Distance` measures forward distance accumulated while carrying;
- `Progressive Carries` counts carries satisfying a separate progression threshold.

The numerator is therefore not the summed distance of only the events counted in the denominator.

The engineered ratio was removed rather than retained under a misleading interpretation.

This illustrates why candidate features are subjected to football and definition-level sanity checks before entering the final style matrix.

In [ ]:
feature_df.drop(
    columns=["avg_progressive_carry_distance"],
    inplace=True
)

carrying_features.remove(
    "avg_progressive_carry_distance"
)

### Progressive Distance per Carry

The initial attempt to estimate average progressive-carry length using:

`Progressive Carrying Distance / Progressive Carries`

was rejected because the numerator and denominator are based on different underlying constructions.

A more defensible distance-based feature is:

$$
Progressive\ Distance\ per\ Carry
=
\frac{Progressive\ Carrying\ Distance}
{Total\ Carries}
$$

This measures the average amount of forward distance generated across the player's overall carrying activity.

Interpretation:

> **How much forward territory does the player typically gain per carry?**

This differs from `progressive_carry_share`:

- `progressive_carry_share` measures how often carries qualify as progressive events;
- `progressive_distance_per_carry` measures how much forward distance is generated across carrying activity.

The feature is retained as **Secondary** because it may still overlap with other carrying-progression measures and can be influenced by role, starting position, and transition context.

In [ ]:
feature_df["progressive_distance_per_carry"] = safe_ratio(
    df["Carries_PrgDist"],
    df["Carries_Carries"]
)

In [ ]:
carrying_features.append(
    "progressive_distance_per_carry"
)

In [ ]:
display(
    feature_df["progressive_distance_per_carry"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

In [ ]:
print("Highest:")
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "progressive_distance_per_carry"
        ]
    ]
    .sort_values(
        "progressive_distance_per_carry",
        ascending=False
    )
    .head(10)
)

print("Lowest:")
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "progressive_distance_per_carry"
        ]
    ]
    .sort_values(
        "progressive_distance_per_carry"
    )
    .head(10)
)

In [ ]:
penetration_features = [
    "carries_into_final_third_p90",
    "carries_into_box_p90"
]

for col in penetration_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

In [ ]:
takeon_features = [
    "take_ons_p90",
    "take_ons_per_100_live_touches"
]

for col in takeon_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

In [ ]:
feature_df["take_ons_p90_pct"] = (
    feature_df["take_ons_p90"]
    .rank(pct=True)
)

feature_df["take_ons_touch_pct"] = (
    feature_df["take_ons_per_100_live_touches"]
    .rank(pct=True)
)

feature_df["take_on_rank_gap"] = (
    feature_df["take_ons_touch_pct"] -
    feature_df["take_ons_p90_pct"]
)

display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "take_ons_p90",
            "take_ons_per_100_live_touches",
            "take_ons_p90_pct",
            "take_ons_touch_pct",
            "take_on_rank_gap"
        ]
    ]
    .sort_values("take_on_rank_gap", ascending=False)
    .head(15)
)

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "take_ons_p90",
            "take_ons_per_100_live_touches",
            "take_ons_p90_pct",
            "take_ons_touch_pct",
            "take_on_rank_gap"
        ]
    ]
    .sort_values("take_on_rank_gap")
    .head(15)
)

### Volume vs Propensity in Take-On Behaviour

The comparison between `take_ons_p90` and `take_ons_per_100_live_touches` confirms that the two measures capture different aspects of dribbling behaviour.

Some players record relatively modest take-on volume per 90 because they are less involved in possession overall, but attempt take-ons very frequently when they do receive the ball. Others accumulate high take-on totals largely within much broader, high-volume possession roles.

This supports retaining both measures at the candidate-feature stage:

- `take_ons_p90` captures the **match-level volume** of 1v1 behaviour;
- `take_ons_per_100_live_touches` captures the **propensity to attempt a take-on when involved**.

The two should therefore not be treated as interchangeable measures.

In [ ]:
feature_df.drop(
    columns=[
        "take_ons_p90_pct",
        "take_ons_touch_pct",
        "take_on_rank_gap"
    ],
    inplace=True
)

# 6. Passing and Progression

Passing behaviour can represent several different aspects of winger style:

1. **general circulation involvement** — how much the player participates in passing;
2. **progressive passing** — how frequently the player advances possession through passes;
3. **territorial penetration** — how frequently passes enter advanced areas or the penalty area;
4. **passing profile** — the types of passes the player tends to choose;
5. **chance creation** — how frequently the player's passes directly lead to shots.

As with carrying and dribbling, match-level volume and behavioural orientation are treated as distinct concepts.

The candidate features in this family are:

| Feature | Status | Interpretation |
|---|---|---|
| `passes_attempted_p90` | Secondary | General passing involvement |
| `progressive_passes_p90` | Core | Match-level progression through passing |
| `progressive_pass_share` | Core | How progressively oriented the player's passing is |
| `passes_into_box_p90` | Core | Frequency of passing possession into the penalty area |
| `passes_into_final_third_p90` | Secondary | Frequency of passing possession into the final third |
| `crosses_p90` | Core | Match-level crossing volume |
| `cross_pass_share` | Core | How cross-oriented the player's passing repertoire is |
| `long_pass_share` | Secondary | Preference for longer distribution |
| `key_passes_p90` | Secondary | Frequency of passes directly leading to shots |

Several of these variables are expected to overlap. They remain in the candidate pool until redundancy is assessed across the complete feature set.

In [ ]:
passing_source_cols = [
    "Passes Attempted",
    "Progressive Passes",
    "Passes into Final 3rd",
    "Passes into Penalty Area",
    "Key passes",
    "Total_Att",
    "Long_Att",
    "Pass Types_Crs"
]

missing_cols = [
    col for col in passing_source_cols
    if col not in df.columns
]

print("Missing passing source columns:", missing_cols)

assert len(missing_cols) == 0

In [ ]:
display(
    df[
        [
            "player",
            "squad",
            "season",
            "Minutes"
        ] + passing_source_cols
    ].sample(10, random_state=42)
)

### Source Alignment for Passing Shares

Where possible, share features are constructed using numerator and denominator statistics from the same source family.

For example:

- `progressive_pass_share` uses the canonical `Progressive Passes` and `Passes Attempted` fields;
- `cross_pass_share` uses donor `Pass Types_Crs` and donor `Total_Att`;
- `long_pass_share` uses donor `Long_Att` and donor `Total_Att`.

This avoids creating ratios from statistics that may have been collected under slightly different historical snapshots or definitions.

In [ ]:
feature_df["passes_attempted_p90"] = per90(
    df["Passes Attempted"],
    df["Minutes"]
)

feature_df["progressive_passes_p90"] = per90(
    df["Progressive Passes"],
    df["Minutes"]
)

feature_df["progressive_pass_share"] = safe_ratio(
    df["Progressive Passes"],
    df["Passes Attempted"]
)

feature_df["passes_into_box_p90"] = per90(
    df["Passes into Penalty Area"],
    df["Minutes"]
)

feature_df["passes_into_final_third_p90"] = per90(
    df["Passes into Final 3rd"],
    df["Minutes"]
)

feature_df["crosses_p90"] = per90(
    df["Pass Types_Crs"],
    df["Minutes"]
)

feature_df["cross_pass_share"] = safe_ratio(
    df["Pass Types_Crs"],
    df["Total_Att"]
)

feature_df["long_pass_share"] = safe_ratio(
    df["Long_Att"],
    df["Total_Att"]
)

feature_df["key_passes_p90"] = per90(
    df["Key passes"],
    df["Minutes"]
)

In [ ]:
passing_features = [
    "passes_attempted_p90",
    "progressive_passes_p90",
    "progressive_pass_share",
    "passes_into_box_p90",
    "passes_into_final_third_p90",
    "crosses_p90",
    "cross_pass_share",
    "long_pass_share",
    "key_passes_p90"
]

In [ ]:
display(
    feature_df[passing_features]
    .describe()
    .T
)

In [ ]:
display(
    feature_df[passing_features]
    .isna()
    .sum()
    .to_frame("missing")
)

In [ ]:
print(
    "Rows with zero canonical pass attempts:",
    (df["Passes Attempted"] == 0).sum()
)

print(
    "Rows with zero donor pass attempts:",
    (df["Total_Att"] == 0).sum()
)

In [ ]:
share_features = [
    "progressive_pass_share",
    "cross_pass_share",
    "long_pass_share"
]

for col in share_features:
    invalid = (
        (feature_df[col] < 0) |
        (feature_df[col] > 1)
    )

    print(
        col,
        "invalid values:",
        invalid.sum()
    )

In [ ]:
progression_passing_features = [
    "passes_attempted_p90",
    "progressive_passes_p90",
    "progressive_pass_share"
]

for col in progression_passing_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

In [ ]:
territorial_passing_features = [
    "passes_into_final_third_p90",
    "passes_into_box_p90"
]

for col in territorial_passing_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

In [ ]:
passing_profile_features = [
    "crosses_p90",
    "cross_pass_share",
    "long_pass_share",
    "key_passes_p90"
]

for col in passing_profile_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest:")
    display(
        feature_df[["player","squad","season","Minutes",col]]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest:")
    display(
        feature_df[["player","squad","season","Minutes",col]]
        .sort_values(col)
        .head(10)
    )

In [ ]:
feature_df["prgp_p90_pct"] = (
    feature_df["progressive_passes_p90"]
    .rank(pct=True)
)

feature_df["prgp_share_pct"] = (
    feature_df["progressive_pass_share"]
    .rank(pct=True)
)

feature_df["progressive_pass_rank_gap"] = (
    feature_df["prgp_share_pct"] -
    feature_df["prgp_p90_pct"]
)

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "progressive_passes_p90",
            "progressive_pass_share",
            "prgp_p90_pct",
            "prgp_share_pct",
            "progressive_pass_rank_gap"
        ]
    ]
    .sort_values(
        "progressive_pass_rank_gap",
        ascending=False
    )
    .head(15)
)

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "progressive_passes_p90",
            "progressive_pass_share",
            "prgp_p90_pct",
            "prgp_share_pct",
            "progressive_pass_rank_gap"
        ]
    ]
    .sort_values("progressive_pass_rank_gap")
    .head(15)
)

In [ ]:
feature_df.drop(
    columns=[
        "prgp_p90_pct",
        "prgp_share_pct",
        "progressive_pass_rank_gap"
    ],
    inplace=True
)

In [ ]:
feature_df["cross_p90_pct"] = (
    feature_df["crosses_p90"]
    .rank(pct=True)
)

feature_df["cross_share_pct"] = (
    feature_df["cross_pass_share"]
    .rank(pct=True)
)

feature_df["cross_rank_gap"] = (
    feature_df["cross_share_pct"] -
    feature_df["cross_p90_pct"]
)

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "crosses_p90",
            "cross_pass_share",
            "cross_p90_pct",
            "cross_share_pct",
            "cross_rank_gap"
        ]
    ]
    .sort_values("cross_rank_gap", ascending=False)
    .head(15)
)

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "crosses_p90",
            "cross_pass_share",
            "cross_p90_pct",
            "cross_share_pct",
            "cross_rank_gap"
        ]
    ]
    .sort_values("cross_rank_gap")
    .head(15)
)

In [ ]:
feature_df.drop(
    columns=[
        "cross_p90_pct",
        "cross_share_pct",
        "cross_rank_gap"
    ],
    inplace=True
)

# 7. Shooting Behaviour

Shooting behaviour is separated from finishing quality.

The purpose of this feature family is to describe:

1. **shooting volume** — how frequently the player chooses to shoot;
2. **shooting propensity** — how shot-oriented the player is when involved in advanced areas;
3. **shot selection** — the typical locations and quality of the shots he chooses or receives.

Finishing outcomes such as goals, shot-on-target percentage and goals per shot are deliberately excluded from the initial style matrix.

This allows two players with similar shooting behaviour to remain stylistically similar even if one is a substantially better finisher.

The candidate features are:

| Feature | Status | Interpretation |
|---|---|---|
| `shots_p90` | Core | Match-level shooting volume |
| `shots_per_100_att3rd_touches` | Core | Shooting propensity relative to advanced involvement |
| `avg_shot_distance` | Core | Typical shooting range |
| `npxg_per_shot` | Secondary | Typical quality of non-penalty shots |

`npxg_per_shot` is retained as Secondary because it reflects both behavioural shot selection and the quality of attacking opportunities created by the player's team.

In [ ]:
shooting_source_cols = [
    "Total Shots",
    "Touches_Att 3rd",
    "Standard_Dist",
    "Expected_npxG/Sh"
]

missing_cols = [
    col for col in shooting_source_cols
    if col not in df.columns
]

print("Missing shooting source columns:", missing_cols)

assert len(missing_cols) == 0

In [ ]:
display(
    df[
        [
            "player",
            "squad",
            "season",
            "Minutes"
        ] + shooting_source_cols
    ].sample(10, random_state=42)
)

In [ ]:
feature_df["shots_p90"] = per90(
    df["Total Shots"],
    df["Minutes"]
)

feature_df["shots_per_100_att3rd_touches"] = per_100(
    df["Total Shots"],
    df["Touches_Att 3rd"]
)

feature_df["avg_shot_distance"] = df["Standard_Dist"]

feature_df["npxg_per_shot"] = df["Expected_npxG/Sh"]

In [ ]:
shooting_features = [
    "shots_p90",
    "shots_per_100_att3rd_touches",
    "avg_shot_distance",
    "npxg_per_shot"
]

In [ ]:
display(
    feature_df[shooting_features]
    .describe()
    .T
)

In [ ]:
display(
    feature_df[shooting_features]
    .isna()
    .sum()
    .to_frame("missing")
)

In [ ]:
print(
    "Rows with zero attacking-third touches:",
    (df["Touches_Att 3rd"] == 0).sum()
)

print(
    "Rows with zero shots:",
    (df["Total Shots"] == 0).sum()
)

In [ ]:
display(
    feature_df[shooting_features]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

In [ ]:
for col in shooting_features:

    print("\n" + "=" * 80)
    print(col.upper())

    print("\nHighest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print("\nLowest:")
    display(
        feature_df[
            [
                "player",
                "squad",
                "season",
                "Minutes",
                col
            ]
        ]
        .sort_values(col)
        .head(10)
    )

In [ ]:
feature_df["shots_p90_pct"] = (
    feature_df["shots_p90"]
    .rank(pct=True)
)

feature_df["shot_propensity_pct"] = (
    feature_df["shots_per_100_att3rd_touches"]
    .rank(pct=True)
)

feature_df["shooting_rank_gap"] = (
    feature_df["shot_propensity_pct"]
    -
    feature_df["shots_p90_pct"]
)

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "shots_p90",
            "shots_per_100_att3rd_touches",
            "shots_p90_pct",
            "shot_propensity_pct",
            "shooting_rank_gap"
        ]
    ]
    .sort_values(
        "shooting_rank_gap",
        ascending=False
    )
    .head(15)
)

In [ ]:
display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "shots_p90",
            "shots_per_100_att3rd_touches",
            "shots_p90_pct",
            "shot_propensity_pct",
            "shooting_rank_gap"
        ]
    ]
    .sort_values("shooting_rank_gap")
    .head(15)
)

In [ ]:
feature_df.drop(
    columns=[
        "shots_p90_pct",
        "shot_propensity_pct",
        "shooting_rank_gap"
    ],
    inplace=True
)

In [ ]:
feature_df[
    [
        "avg_shot_distance",
        "npxg_per_shot"
    ]
].corr()

### Shooting Feature Sanity Check

Inspection of the shooting features showed that the engineered variables produce football-interpretable player profiles.

`shots_p90` and `shots_per_100_att3rd_touches` capture related but distinct concepts:

- `shots_p90` measures actual match-level shooting volume;
- `shots_per_100_att3rd_touches` measures how shot-oriented a player's advanced involvement is.

`avg_shot_distance` provides a direct description of shooting range, while `npxg_per_shot` captures the average quality of non-penalty shooting opportunities.

The latter remains classified as Secondary because shot quality reflects both player behaviour and the attacking opportunities generated by team context.

All four variables remain in the candidate pool pending later redundancy and temporal-stability analysis.

# 8. Cross-Family Behavioural Choice

Most features constructed so far measure the volume or orientation of a single behaviour.

This section instead compares **alternative methods of achieving the same attacking objective**.

Two candidate features are constructed:

### Carry Progression Share

$$
Carry\ Progression\ Share
=
\frac{Progressive\ Carries}
{Progressive\ Carries + Progressive\ Passes}
$$

Interpretation:

> When the player progresses possession through either carrying or passing, what proportion of that progression comes through carrying?

A high value indicates a more carry-oriented progression profile.

A low value indicates a more pass-oriented progression profile.

---

### Carry Box-Entry Share

$$
Carry\ BoxEntry\ Share
=
\frac{Carries\ into\ Penalty\ Area}
{Carries\ into\ Penalty\ Area + Passes\ into\ Penalty\ Area}
$$

Interpretation:

> When the player penetrates the penalty area through either carrying or passing, how often does he enter the box himself rather than deliver the ball into it?

A high value indicates a more self-penetrative ball-carrying profile.

A low value indicates a more supply-oriented passing profile.

These features are classified as **Core** because they directly describe relative attacking preferences rather than absolute activity volume.

However, they should not be interpreted as complete measures of progression or box entry. They only compare the two observable mechanisms available in the dataset.

In [ ]:
choice_source_cols = [
    "Progressive Carries",
    "Progressive Passes",
    "Carries into Penalty Area",
    "Passes into Penalty Area"
]

missing_cols = [
    col for col in choice_source_cols
    if col not in df.columns
]

print("Missing cross-family source columns:", missing_cols)

assert len(missing_cols) == 0

In [ ]:
display(
    df[
        [
            "player",
            "squad",
            "season",
            "Minutes"
        ] + choice_source_cols
    ].sample(10, random_state=42)
)

In [ ]:
feature_df["carry_progression_share"] = safe_ratio(
    df["Progressive Carries"],
    df["Progressive Carries"] + df["Progressive Passes"]
)

feature_df["carry_box_entry_share"] = safe_ratio(
    df["Carries into Penalty Area"],
    df["Carries into Penalty Area"] + df["Passes into Penalty Area"]
)

In [ ]:
choice_features = [
    "carry_progression_share",
    "carry_box_entry_share"
]

In [ ]:
progression_denominator = (
    df["Progressive Carries"] +
    df["Progressive Passes"]
)

box_entry_denominator = (
    df["Carries into Penalty Area"] +
    df["Passes into Penalty Area"]
)

print(
    "Rows with zero measurable progression actions:",
    (progression_denominator == 0).sum()
)

print(
    "Rows with zero measurable box-entry actions:",
    (box_entry_denominator == 0).sum()
)

In [ ]:
display(
    feature_df[choice_features]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

In [ ]:
display(
    feature_df[choice_features]
    .isna()
    .sum()
    .to_frame("missing")
)

In [ ]:
for col in choice_features:

    invalid = (
        (feature_df[col] < 0) |
        (feature_df[col] > 1)
    )

    print(
        col,
        "invalid values:",
        invalid.sum()
    )

In [ ]:
print("Highest carry progression share:")

display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "carry_progression_share"
        ]
    ]
    .sort_values(
        "carry_progression_share",
        ascending=False
    )
    .head(15)
)

print("Lowest carry progression share:")

display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "carry_progression_share"
        ]
    ]
    .sort_values("carry_progression_share")
    .head(15)
)

In [ ]:
print("Highest carry box-entry share:")

display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "carry_box_entry_share"
        ]
    ]
    .sort_values(
        "carry_box_entry_share",
        ascending=False
    )
    .head(15)
)

print("Lowest carry box-entry share:")

display(
    feature_df[
        [
            "player",
            "squad",
            "season",
            "Minutes",
            "carry_box_entry_share"
        ]
    ]
    .sort_values("carry_box_entry_share")
    .head(15)
)

In [ ]:
choice_diagnostic = feature_df[
    [
        "player",
        "squad",
        "season",
        "Minutes",
        "carry_progression_share",
        "carry_box_entry_share"
    ]
].copy()

choice_diagnostic["progression_actions"] = progression_denominator
choice_diagnostic["box_entry_actions"] = box_entry_denominator

In [ ]:
display(
    choice_diagnostic[
        [
            "player",
            "squad",
            "season",
            "carry_box_entry_share",
            "box_entry_actions"
        ]
    ]
    .sort_values(
        "carry_box_entry_share",
        ascending=False
    )
    .head(20)
)

In [ ]:
display(
    choice_diagnostic[
        [
            "player",
            "squad",
            "season",
            "carry_progression_share",
            "progression_actions"
        ]
    ]
    .sort_values(
        "carry_progression_share",
        ascending=False
    )
    .head(20)
)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    df["Progressive Passes"],
    df["Progressive Carries"],
    alpha=0.5
)

plt.xlabel("Progressive Passes")
plt.ylabel("Progressive Carries")
plt.title("Progressive Passing vs Progressive Carrying")

plt.show()

In [ ]:
display(
    pd.DataFrame({
        "player": df["player"],
        "squad": df["squad"],
        "season": df["season"],

        "PrgC": df["Progressive Carries"],
        "PrgP": df["Progressive Passes"],
        "carry_progression_share":
            feature_df["carry_progression_share"],

        "CPA": df["Carries into Penalty Area"],
        "PPA": df["Passes into Penalty Area"],
        "carry_box_entry_share":
            feature_df["carry_box_entry_share"]
    })
    .sort_values(
        "carry_progression_share",
        ascending=False
    )
    .head(20)
)

### Cross-Family Behavioural Choice Sanity Check

The cross-family ratios provide a different type of information from the preceding volume and propensity features.

`carry_progression_share` compares two alternative methods of advancing possession:

- carrying;
- passing.

`carry_box_entry_share` compares two alternative methods of penetrating the penalty area:

- entering with the ball;
- delivering the ball through a pass.

These measures are therefore interpreted as **relative behavioural preferences**, not as measures of total attacking involvement.

Extreme values are interpreted alongside their underlying action counts because ratios based on very small denominators may be unstable.

No minimum action threshold is imposed at this stage. Denominator reliability will be reconsidered during the complete candidate-feature quality audit.

# 9. Supporting Descriptors

The primary feature matrix is designed to answer:

> **How does the player tend to attack?**

A separate set of supporting descriptors is now constructed to help answer:

> **How effectively does that style work, and what other characteristics accompany it?**

These variables are deliberately excluded from initial style discovery.

They include measures of:

- 1v1 execution;
- ball security;
- passing execution;
- chance-creation effectiveness;
- attacking output;
- aerial involvement;
- high-pitch defensive involvement.

Keeping these variables separate allows stylistically similar players to be compared on quality and effectiveness after the style structure has been discovered.

For example, two players may have similar dribbling and progression profiles while differing substantially in take-on success, chance creation or finishing output.

In [ ]:
support_df = df[identity_cols].copy()

print(support_df.shape)

display(
    support_df.head()
)

## 9.1 Dribbling Execution and Ball Security

Take-on frequency belongs to the playstyle representation because it describes the player's tendency to attack defenders.

Take-on success, miscontrols and dispossessions instead describe how successfully or securely those behaviours are executed.

These variables are therefore retained as supporting descriptors rather than style features.

In [ ]:
support_df["take_on_success_pct"] = pd.to_numeric(
    df["Successful Take-ons %"],
    errors="coerce"
)

support_df["miscontrols_p90"] = per90(
    df["Carries_Mis"],
    df["Minutes"]
)

support_df["dispossessed_p90"] = per90(
    df["Times Dispossessed"],
    df["Minutes"]
)

## 9.2 Passing Execution

Passing completion is retained as an execution descriptor rather than a style feature.

A player attempting more aggressive or difficult passes may naturally record lower completion rates than a player making safer choices.

Completion percentage should therefore help evaluate a passing style rather than define it.

In [ ]:
support_df["pass_completion_pct"] = pd.to_numeric(
    df["Pass Completion %"],
    errors="coerce"
)

support_df["long_pass_completion_pct"] = pd.to_numeric(
    df["Long Pass Completion %"],
    errors="coerce"
)

## 9.3 Chance Creation and Attacking Value

Several variables describe the effectiveness or downstream value of attacking actions.

These include:

- expected assists;
- expected assisted goals;
- shot-creating actions.

They are not used to define style because they depend partly on the quality of the resulting chance and on subsequent actions by teammates.

Instead, they are retained to evaluate how productive different attacking styles are.

In [ ]:
support_df["xa_p90"] = per90(
    df["Expected_xA"],
    df["Minutes"]
)

support_df["xag_p90"] = per90(
    df["Expected_xAG"],
    df["Minutes"]
)

support_df["sca_p90"] = pd.to_numeric(
    df["Shot Creating Actions per 90"],
    errors="coerce"
)

## 9.4 Scoring and Attacking Output

Final attacking output is deliberately excluded from style discovery.

However, output variables are retained so that discovered styles can later be evaluated on production.

The supporting output layer focuses on non-penalty scoring, expected scoring and assists.

In [ ]:
support_df["non_penalty_goals_p90"] = per90(
    df["Non-Penalty Goals"],
    df["Minutes"]
)

support_df["npxg_p90"] = per90(
    df["Non-Penalty Expected Goals"],
    df["Minutes"]
)

support_df["assists_p90"] = per90(
    df["Assists"],
    df["Minutes"]
)

## 9.5 Aerial Involvement

Aerial involvement can describe a meaningful secondary role difference between wide players.

Some wingers frequently contest aerial balls, act as direct outlets or attack aerial deliveries, while others receive almost entirely on the ground.

Aerial duel frequency is therefore retained as a supplemental role descriptor.

Aerial win percentage remains an execution measure rather than a style feature.

In [ ]:
aerial_duels = (
    df["Aerial Duels_Won"] +
    df["Aerial Duels_Lost"]
)

support_df["aerial_duels_p90"] = per90(
    aerial_duels,
    df["Minutes"]
)

support_df["aerial_win_pct"] = (
    safe_ratio(
        df["Aerial Duels_Won"],
        aerial_duels
    ) * 100
)

## 9.6 High-Pitch Defensive Involvement

The primary model focuses on attacking winger style.

However, one defensive descriptor is retained:

`attacking_third_tackles_p90`

This measures defensive interventions made in the attacking third.

It should not be interpreted directly as pressing intensity because a press does not necessarily result in a tackle, and tackles can occur outside coordinated pressing structures.

The feature is therefore interpreted more conservatively as:

> **high-pitch defensive intervention frequency**

In [ ]:
support_df["attacking_third_tackles_p90"] = per90(
    df["Tackles_Att 3rd"],
    df["Minutes"]
)

In [ ]:
support_features = [
    # Dribbling execution / ball security
    "take_on_success_pct",
    "miscontrols_p90",
    "dispossessed_p90",

    # Passing execution
    "pass_completion_pct",
    "long_pass_completion_pct",

    # Chance creation
    "xa_p90",
    "xag_p90",
    "sca_p90",

    # Output
    "non_penalty_goals_p90",
    "npxg_p90",
    "assists_p90",

    # Supplemental role
    "aerial_duels_p90",
    "aerial_win_pct",
    "attacking_third_tackles_p90"
]

print("Supporting descriptors:", len(support_features))

In [ ]:
display(
    support_df[support_features]
    .describe()
    .T
)

In [ ]:
support_missing = (
    support_df[support_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing")
)

support_missing["missing_pct"] = (
    support_missing["missing"] /
    len(support_df) *
    100
)

display(support_missing)

In [ ]:
percentage_features = [
    "take_on_success_pct",
    "pass_completion_pct",
    "long_pass_completion_pct",
    "aerial_win_pct"
]

for col in percentage_features:

    print(
        col,
        "min:",
        support_df[col].min(),
        "max:",
        support_df[col].max()
    )

In [ ]:
print(
    "Rows with zero aerial duels:",
    (aerial_duels == 0).sum()
)

In [ ]:
descriptor_sanity_features = [
    "take_on_success_pct",
    "dispossessed_p90",
    "xag_p90",
    "non_penalty_goals_p90",
    "aerial_duels_p90",
    "attacking_third_tackles_p90"
]

for col in descriptor_sanity_features:

    print("\n" + "=" * 80)
    print(col.upper())

    display(
        support_df[
            [
                "player",
                "squad",
                "season",
                col
            ]
        ]
        .sort_values(
            col,
            ascending=False
        )
        .head(10)
    )

### Supporting Descriptor Status

A compact supporting layer has now been constructed alongside the candidate attacking-style matrix.

The two datasets serve different analytical purposes:

### `feature_df`

Contains behavioural features that may eventually drive:

- player similarity;
- dimensionality reduction;
- archetype discovery.

### `support_df`

Contains quality, output and supplemental role descriptors used to:

- evaluate discovered styles;
- explain differences between stylistically similar players;
- support recruitment and role-fit interpretation.

Supporting descriptors are not included in the initial style-discovery feature space.

# 10. Candidate Feature Quality Audit

Feature construction does not guarantee that a variable is suitable for modelling.

The complete candidate pool is therefore audited before any standardization, dimensionality reduction or similarity analysis.

This stage asks:

- Are values missing?
- Are any features effectively constant?
- Are distributions excessively sparse or skewed?
- Are engineered shares bounded as expected?
- Are extreme values plausible?
- Are ratio features being driven by very small denominators?
- Do any features show evidence of data-definition instability?

Importantly, no feature is removed simply because its distribution is unusual.

Football behaviours are often naturally skewed.

A feature is only reconsidered when its statistical behaviour creates a methodological or interpretability problem.

## 10.1 Candidate Feature Registry

All attacking-style candidates constructed so far are consolidated into a single registry.

The registry preserves each feature's:

- conceptual family;
- current classification as Core or Secondary;
- intended role in the analysis.

These classifications remain provisional until the empirical audit is complete.

In [ ]:
feature_registry = pd.DataFrame([
    # Territory
    ("attacking_third_touch_share", "Territory", "Core"),
    ("box_penetration_touch_share", "Territory", "Core"),
    ("box_touch_share", "Territory", "Secondary"),

    # Receiving
    ("progressive_receptions_p90", "Receiving", "Core"),
    ("progressive_reception_share", "Receiving", "Core"),

    # Carrying / Dribbling
    ("progressive_carries_p90", "Carrying", "Core"),
    ("progressive_carry_share", "Carrying", "Core"),
    ("carries_into_box_p90", "Carrying", "Core"),
    ("carries_into_final_third_p90", "Carrying", "Secondary"),
    ("take_ons_p90", "Dribbling", "Core"),
    ("take_ons_per_100_live_touches", "Dribbling", "Core"),
    ("progressive_distance_per_carry", "Carrying", "Secondary"),

    # Passing / Progression
    ("passes_attempted_p90", "Passing", "Secondary"),
    ("progressive_passes_p90", "Passing", "Core"),
    ("progressive_pass_share", "Passing", "Core"),
    ("passes_into_box_p90", "Passing", "Core"),
    ("passes_into_final_third_p90", "Passing", "Secondary"),
    ("crosses_p90", "Crossing", "Core"),
    ("cross_pass_share", "Crossing", "Core"),
    ("long_pass_share", "Passing Profile", "Secondary"),
    ("key_passes_p90", "Creation", "Secondary"),

    # Shooting
    ("shots_p90", "Shooting", "Core"),
    ("shots_per_100_att3rd_touches", "Shooting", "Core"),
    ("avg_shot_distance", "Shooting", "Core"),
    ("npxg_per_shot", "Shooting", "Secondary"),

    # Cross-family
    ("carry_progression_share", "Progression Method", "Core"),
    ("carry_box_entry_share", "Penetration Method", "Core"),
], columns=[
    "feature",
    "family",
    "status"
])

print("Candidate features:", len(feature_registry))

display(feature_registry)

In [ ]:
candidate_features = feature_registry["feature"].tolist()

missing_engineered_features = [
    col for col in candidate_features
    if col not in feature_df.columns
]

print(
    "Missing engineered candidate features:",
    missing_engineered_features
)

assert len(missing_engineered_features) == 0

In [ ]:
qa_rows = []

for col in candidate_features:

    s = feature_df[col]

    qa_rows.append({
        "feature": col,
        "missing_n": s.isna().sum(),
        "missing_pct": s.isna().mean() * 100,
        "unique_n": s.nunique(dropna=True),
        "zero_n": (s == 0).sum(),
        "zero_pct": (s == 0).mean() * 100,
        "min": s.min(),
        "q01": s.quantile(0.01),
        "q05": s.quantile(0.05),
        "median": s.median(),
        "q95": s.quantile(0.95),
        "q99": s.quantile(0.99),
        "max": s.max(),
        "mean": s.mean(),
        "std": s.std(),
        "skew": s.skew()
    })

feature_qa = pd.DataFrame(qa_rows)

feature_qa = feature_qa.merge(
    feature_registry,
    on="feature",
    how="left",
    validate="one_to_one"
)

feature_qa = feature_qa[
    [
        "feature",
        "family",
        "status",
        "missing_n",
        "missing_pct",
        "unique_n",
        "zero_n",
        "zero_pct",
        "min",
        "q01",
        "q05",
        "median",
        "q95",
        "q99",
        "max",
        "mean",
        "std",
        "skew"
    ]
]

display(feature_qa)

## 10.3 Missingness

Missingness is evaluated before modelling because missing values may arise for different reasons:

- source-data absence;
- undefined ratios caused by zero denominators;
- genuine absence of the underlying behaviour;
- historical differences in data availability.

Missingness is therefore interpreted feature-by-feature rather than automatically imputed.

In [ ]:
missing_audit = (
    feature_qa[
        [
            "feature",
            "family",
            "status",
            "missing_n",
            "missing_pct"
        ]
    ]
    .sort_values(
        "missing_pct",
        ascending=False
    )
)

display(missing_audit)

In [ ]:
display(
    missing_audit[
        missing_audit["missing_n"] > 0
    ]
)

## 10.4 Zero Concentration and Information Content

Some football behaviours may naturally occur infrequently.

A high proportion of zero observations is therefore not automatically grounds for exclusion.

However, extreme zero concentration can make a feature:

- unstable;
- dominated by rare events;
- less useful for similarity modelling.

Zero rates and the number of distinct values are inspected as indicators of potential low-information features.

In [ ]:
zero_audit = (
    feature_qa[
        [
            "feature",
            "family",
            "status",
            "unique_n",
            "zero_n",
            "zero_pct"
        ]
    ]
    .sort_values(
        "zero_pct",
        ascending=False
    )
)

display(zero_audit)

In [ ]:
potentially_sparse = feature_qa[
    (feature_qa["zero_pct"] >= 25) |
    (feature_qa["unique_n"] <= 20)
][
    [
        "feature",
        "family",
        "status",
        "unique_n",
        "zero_pct"
    ]
]

display(potentially_sparse)

In [ ]:
share_candidates = [
    "attacking_third_touch_share",
    "box_penetration_touch_share",
    "box_touch_share",
    "progressive_reception_share",
    "progressive_carry_share",
    "progressive_pass_share",
    "cross_pass_share",
    "long_pass_share",
    "carry_progression_share",
    "carry_box_entry_share"
]

for col in share_candidates:

    invalid = (
        feature_df[col].notna() &
        (
            (feature_df[col] < 0) |
            (feature_df[col] > 1)
        )
    )

    print(
        f"{col}: {invalid.sum()} invalid values"
    )

In [ ]:
negative_counts = {}

for col in candidate_features:

    negative_counts[col] = (
        feature_df[col] < 0
    ).sum()

negative_counts = pd.Series(
    negative_counts,
    name="negative_values"
)

display(
    negative_counts[
        negative_counts > 0
    ]
)

In [ ]:
def plot_feature_distributions(
    dataframe,
    features,
    bins=30,
    figsize=(12, 3)
):
    """
    Plot one histogram per feature.
    """

    for col in features:

        plt.figure(figsize=figsize)

        plt.hist(
            dataframe[col].dropna(),
            bins=bins
        )

        plt.title(col)
        plt.xlabel(col)
        plt.ylabel("Observations")

        plt.show()

In [ ]:
plot_feature_distributions(
    feature_df,
    territory_features
)

In [ ]:
plot_feature_distributions(
    feature_df,
    receiving_features
)

In [ ]:
plot_feature_distributions(
    feature_df,
    carrying_features
)

In [ ]:
plot_feature_distributions(
    feature_df,
    passing_features
)

In [ ]:
plot_feature_distributions(
    feature_df,
    shooting_features
)

In [ ]:
plot_feature_distributions(
    feature_df,
    choice_features
)

In [ ]:
skew_audit = (
    feature_qa[
        [
            "feature",
            "family",
            "status",
            "skew"
        ]
    ]
    .assign(
        abs_skew=lambda x: x["skew"].abs()
    )
    .sort_values(
        "abs_skew",
        ascending=False
    )
)

display(skew_audit)

In [ ]:
display(
    skew_audit[
        skew_audit["abs_skew"] >= 2
    ]
)

In [ ]:
extreme_audit_rows = []

for col in candidate_features:

    s = feature_df[col].dropna()

    q99 = s.quantile(0.99)
    maximum = s.max()

    if q99 != 0:
        max_to_q99 = maximum / q99
    else:
        max_to_q99 = np.nan

    extreme_audit_rows.append({
        "feature": col,
        "q99": q99,
        "max": maximum,
        "max_to_q99": max_to_q99
    })

extreme_audit = (
    pd.DataFrame(extreme_audit_rows)
    .sort_values(
        "max_to_q99",
        ascending=False
    )
)

display(extreme_audit)

In [ ]:
display(
    extreme_audit[
        extreme_audit["max_to_q99"] > 1.5
    ]
)

## 10.9 Ratio Denominator Reliability

Ratio features can produce apparently extreme stylistic values from very small underlying action counts.

For example:

- 1 carry into the box;
- 0 passes into the box;

produces a carry box-entry share of 1.0.

The value is mathematically correct, but provides less evidence of a stable behavioural preference than the same ratio based on dozens of actions.

Ratio features are therefore audited alongside their underlying denominators.

No denominator threshold is imposed at this stage.

In [ ]:
denominator_audit = pd.DataFrame({
    "player": df["player"],
    "squad": df["squad"],
    "season": df["season"],

    "total_touches":
        df["Touches_Touches"],

    "attacking_third_touches":
        df["Touches_Att 3rd"],

    "receptions":
        df["Receiving_Rec"],

    "total_carries":
        df["Carries_Carries"],

    "live_touches":
        df["Touches_Live"],

    "pass_attempts":
        df["Passes Attempted"],

    "donor_pass_attempts":
        df["Total_Att"],

    "progression_actions":
        (
            df["Progressive Carries"] +
            df["Progressive Passes"]
        ),

    "box_entry_actions":
        (
            df["Carries into Penalty Area"] +
            df["Passes into Penalty Area"]
        )
})

In [ ]:
display(
    denominator_audit.drop(
        columns=[
            "player",
            "squad",
            "season"
        ]
    )
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50
        ]
    )
    .T
)

In [ ]:
display(
    denominator_audit[
        [
            "progression_actions",
            "box_entry_actions"
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
    .T
)

In [ ]:
for threshold in [1, 3, 5, 10, 20]:

    count = (
        denominator_audit["box_entry_actions"]
        < threshold
    ).sum()

    print(
        f"Rows with fewer than {threshold} "
        f"box-entry actions: {count} "
        f"({count / len(df) * 100:.1f}%)"
    )

In [ ]:
for threshold in [5, 10, 20, 30, 50]:

    count = (
        denominator_audit["progression_actions"]
        < threshold
    ).sum()

    print(
        f"Rows with fewer than {threshold} "
        f"progression actions: {count} "
        f"({count / len(df) * 100:.1f}%)"
    )

In [ ]:
box_entry_diagnostic = pd.DataFrame({
    "player": df["player"],
    "squad": df["squad"],
    "season": df["season"],
    "CPA": df["Carries into Penalty Area"],
    "PPA": df["Passes into Penalty Area"],
    "box_entry_actions":
        denominator_audit["box_entry_actions"],
    "carry_box_entry_share":
        feature_df["carry_box_entry_share"]
})

display(
    box_entry_diagnostic
    .sort_values(
        "box_entry_actions"
    )
    .head(30)
)

In [ ]:
progression_diagnostic = pd.DataFrame({
    "player": df["player"],
    "squad": df["squad"],
    "season": df["season"],
    "PrgC": df["Progressive Carries"],
    "PrgP": df["Progressive Passes"],
    "progression_actions":
        denominator_audit["progression_actions"],
    "carry_progression_share":
        feature_df["carry_progression_share"]
})

display(
    progression_diagnostic
    .sort_values(
        "progression_actions"
    )
    .head(30)
)

In [ ]:
feature_audit_summary = feature_qa.copy()

feature_audit_summary["missing_flag"] = (
    feature_audit_summary["missing_pct"] > 0
)

feature_audit_summary["sparse_flag"] = (
    feature_audit_summary["zero_pct"] >= 25
)

feature_audit_summary["high_skew_flag"] = (
    feature_audit_summary["skew"].abs() >= 2
)

feature_audit_summary = (
    feature_audit_summary.merge(
        extreme_audit[
            [
                "feature",
                "max_to_q99"
            ]
        ],
        on="feature",
        how="left"
    )
)

feature_audit_summary["extreme_flag"] = (
    feature_audit_summary["max_to_q99"] > 1.5
)

In [ ]:
display(
    feature_audit_summary[
        [
            "feature",
            "family",
            "status",
            "missing_pct",
            "zero_pct",
            "skew",
            "max_to_q99",
            "missing_flag",
            "sparse_flag",
            "high_skew_flag",
            "extreme_flag"
        ]
    ]
)

In [ ]:
flagged_features = feature_audit_summary[
    feature_audit_summary[
        [
            "missing_flag",
            "sparse_flag",
            "high_skew_flag",
            "extreme_flag"
        ]
    ].any(axis=1)
]

display(
    flagged_features[
        [
            "feature",
            "family",
            "status",
            "missing_pct",
            "zero_pct",
            "skew",
            "max_to_q99",
            "missing_flag",
            "sparse_flag",
            "high_skew_flag",
            "extreme_flag"
        ]
    ]
)

## 10A. Structural Audit — Findings

The structural feature audit evaluates each candidate for:

- missingness;
- sparsity;
- theoretical validity;
- distribution shape;
- extreme-value concentration;
- ratio-denominator reliability.

Diagnostic flags are not treated as automatic exclusion criteria.

Each flagged variable is reviewed in football and data-definition context before any decision is made.

The next stage examines whether the candidate features are **temporally comparable across seasons**, including potential historical definition changes in the underlying FBref data.

# 10B. Temporal Stability and Historical Definition Audit

The dataset spans seven seasons from 2017-18 to 2023-24.

For a playstyle feature to support cross-season player comparison, its interpretation should remain reasonably stable through time.

However, changes in seasonal distributions can arise from several different causes:

- genuine tactical evolution;
- changes in the winger population;
- league-level changes;
- sample variation;
- changes in the underlying data provider or event definition.

A shift in a seasonal median is therefore **not automatically evidence of a data break**.

The purpose of this section is to identify suspicious discontinuities that require closer investigation, particularly around variables previously flagged as potentially sensitive to historical definition changes.

The audit proceeds in three stages:

1. inspect season-level distributions for every candidate feature;
2. quantify adjacent-season changes;
3. investigate suspicious features at the raw-variable level.

No temporal standardization or correction is applied until the evidence is reviewed.

In [ ]:
season_order = [
    "2017-2018",
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024"
]

print(df["season"].value_counts().sort_index())

unexpected_seasons = set(df["season"].dropna()) - set(season_order)

print("Unexpected seasons:", unexpected_seasons)

assert len(unexpected_seasons) == 0

In [ ]:
season_summary_rows = []

for feature in candidate_features:

    for season in season_order:

        s = feature_df.loc[
            feature_df["season"] == season,
            feature
        ].dropna()

        season_summary_rows.append({
            "feature": feature,
            "season": season,
            "n": len(s),
            "mean": s.mean(),
            "median": s.median(),
            "q25": s.quantile(0.25),
            "q75": s.quantile(0.75),
            "std": s.std(),
            "missing_pct": (
                feature_df.loc[
                    feature_df["season"] == season,
                    feature
                ].isna().mean() * 100
            )
        })

season_summary = pd.DataFrame(season_summary_rows)

season_summary = season_summary.merge(
    feature_registry,
    on="feature",
    how="left",
    validate="many_to_one"
)

display(season_summary.head(15))

In [ ]:
def plot_season_medians(summary_df, features):
    for feature in features:

        temp = (
            summary_df[
                summary_df["feature"] == feature
            ]
            .set_index("season")
            .reindex(season_order)
            .reset_index()
        )

        plt.figure(figsize=(9, 4))

        plt.plot(
            temp["season"],
            temp["median"],
            marker="o"
        )

        plt.title(feature)
        plt.xlabel("Season")
        plt.ylabel("Season median")
        plt.xticks(rotation=45)

        plt.show()

In [ ]:
plot_season_medians(
    season_summary,
    territory_features
)

In [ ]:
plot_season_medians(
    season_summary,
    receiving_features
)

In [ ]:
plot_season_medians(
    season_summary,
    carrying_features
)

In [ ]:
plot_season_medians(
    season_summary,
    passing_features
)

In [ ]:
plot_season_medians(
    season_summary,
    shooting_features
)

In [ ]:
plot_season_medians(
    season_summary,
    choice_features
)

In [ ]:
def plot_season_distribution_band(summary_df, feature):

    temp = (
        summary_df[
            summary_df["feature"] == feature
        ]
        .set_index("season")
        .reindex(season_order)
        .reset_index()
    )

    x = np.arange(len(temp))

    plt.figure(figsize=(9, 5))

    plt.plot(
        x,
        temp["median"],
        marker="o",
        label="Median"
    )

    plt.fill_between(
        x,
        temp["q25"],
        temp["q75"],
        alpha=0.2,
        label="IQR"
    )

    plt.xticks(
        x,
        temp["season"],
        rotation=45
    )

    plt.title(feature)
    plt.xlabel("Season")
    plt.ylabel(feature)
    plt.legend()

    plt.show()

In [ ]:
season_change_rows = []

for feature in candidate_features:

    temp = (
        season_summary[
            season_summary["feature"] == feature
        ]
        .set_index("season")
        .reindex(season_order)
    )

    medians = temp["median"]

    for i in range(1, len(season_order)):

        previous_season = season_order[i - 1]
        current_season = season_order[i]

        prev_median = medians.loc[previous_season]
        curr_median = medians.loc[current_season]

        if pd.notna(prev_median) and prev_median != 0:
            relative_change = (
                (curr_median - prev_median) /
                abs(prev_median)
            )
        else:
            relative_change = np.nan

        season_change_rows.append({
            "feature": feature,
            "from_season": previous_season,
            "to_season": current_season,
            "previous_median": prev_median,
            "current_median": curr_median,
            "relative_median_change": relative_change
        })

season_changes = pd.DataFrame(season_change_rows)

season_changes = season_changes.merge(
    feature_registry,
    on="feature",
    how="left",
    validate="many_to_one"
)

In [ ]:
display(
    season_changes.assign(
        abs_change=lambda x:
            x["relative_median_change"].abs()
    )
    .sort_values(
        "abs_change",
        ascending=False
    )
    .head(40)
)

In [ ]:
robust_shift_rows = []

for feature in candidate_features:

    temp = (
        season_summary[
            season_summary["feature"] == feature
        ]
        .set_index("season")
        .reindex(season_order)
    )

    for i in range(1, len(season_order)):

        s1 = season_order[i - 1]
        s2 = season_order[i]

        m1 = temp.loc[s1, "median"]
        m2 = temp.loc[s2, "median"]

        iqr1 = (
            temp.loc[s1, "q75"] -
            temp.loc[s1, "q25"]
        )

        iqr2 = (
            temp.loc[s2, "q75"] -
            temp.loc[s2, "q25"]
        )

        pooled_iqr = np.nanmean([
            iqr1,
            iqr2
        ])

        if (
            pd.notna(pooled_iqr) and
            pooled_iqr > 0
        ):
            robust_shift = (
                (m2 - m1) /
                pooled_iqr
            )
        else:
            robust_shift = np.nan

        robust_shift_rows.append({
            "feature": feature,
            "from_season": s1,
            "to_season": s2,
            "robust_shift": robust_shift
        })

robust_shifts = pd.DataFrame(
    robust_shift_rows
)

season_changes = season_changes.merge(
    robust_shifts,
    on=[
        "feature",
        "from_season",
        "to_season"
    ],
    how="left",
    validate="one_to_one"
)

In [ ]:
display(
    season_changes.assign(
        abs_robust_shift=lambda x:
            x["robust_shift"].abs()
    )
    .sort_values(
        "abs_robust_shift",
        ascending=False
    )
    .head(40)
)

In [ ]:
suspected_break = season_changes[
    (season_changes["from_season"] == "2021-2022") &
    (season_changes["to_season"] == "2022-2023")
].copy()

suspected_break["abs_robust_shift"] = (
    suspected_break["robust_shift"].abs()
)

display(
    suspected_break[
        [
            "feature",
            "family",
            "status",
            "previous_median",
            "current_median",
            "relative_median_change",
            "robust_shift",
            "abs_robust_shift"
        ]
    ]
    .sort_values(
        "abs_robust_shift",
        ascending=False
    )
)

In [ ]:
temporal_flags = suspected_break.copy()

temporal_flags["large_relative_change"] = (
    temporal_flags[
        "relative_median_change"
    ].abs() >= 0.25
)

temporal_flags["large_robust_shift"] = (
    temporal_flags[
        "robust_shift"
    ].abs() >= 0.5
)

temporal_flags["suspected_break_flag"] = (
    temporal_flags[
        [
            "large_relative_change",
            "large_robust_shift"
        ]
    ].all(axis=1)
)

display(
    temporal_flags[
        [
            "feature",
            "family",
            "status",
            "relative_median_change",
            "robust_shift",
            "large_relative_change",
            "large_robust_shift",
            "suspected_break_flag"
        ]
    ]
    .sort_values(
        "robust_shift",
        key=lambda x: x.abs(),
        ascending=False
    )
)

In [ ]:
takeon_season_audit = (
    df.groupby("season")
    .agg(
        original_takeons_median=(
            "Take-ons Attempted",
            "median"
        ),
        donor_takeons_median=(
            "Take-Ons_Att",
            "median"
        ),
        original_takeons_mean=(
            "Take-ons Attempted",
            "mean"
        ),
        donor_takeons_mean=(
            "Take-Ons_Att",
            "mean"
        ),
        observations=(
            "player",
            "size"
        )
    )
    .reindex(season_order)
)

display(takeon_season_audit)

In [ ]:
takeon_temporal = pd.DataFrame({
    "season": df["season"],

    "original_takeons_p90":
        per90(
            df["Take-ons Attempted"],
            df["Minutes"]
        ),

    "donor_takeons_p90":
        per90(
            df["Take-Ons_Att"],
            df["Minutes"]
        )
})

In [ ]:
display(
    takeon_temporal
    .groupby("season")
    [
        [
            "original_takeons_p90",
            "donor_takeons_p90"
        ]
    ]
    .median()
    .reindex(season_order)
)

In [ ]:
takeon_source_medians = (
    takeon_temporal
    .groupby("season")
    [
        [
            "original_takeons_p90",
            "donor_takeons_p90"
        ]
    ]
    .median()
    .reindex(season_order)
)

plt.figure(figsize=(9, 5))

plt.plot(
    season_order,
    takeon_source_medians[
        "original_takeons_p90"
    ],
    marker="o",
    label="Canonical source"
)

plt.plot(
    season_order,
    takeon_source_medians[
        "donor_takeons_p90"
    ],
    marker="o",
    label="Donor source"
)

plt.xticks(rotation=45)
plt.ylabel("Median take-ons per 90")
plt.title(
    "Take-On Volume Across Seasons"
)
plt.legend()

plt.show()

In [ ]:
takeon_quality_temporal = pd.DataFrame({
    "season": df["season"],

    "canonical_success_pct":
        pd.to_numeric(
            df["Successful Take-ons %"],
            errors="coerce"
        ),

    "donor_success_pct":
        pd.to_numeric(
            df["Take-Ons_Succ%"],
            errors="coerce"
        ),

    "canonical_tackled_pct":
        pd.to_numeric(
            df["Tackled During Take-on %"],
            errors="coerce"
        ),

    "donor_tackled_pct":
        pd.to_numeric(
            df["Take-Ons_Tkld%"],
            errors="coerce"
        )
})

In [ ]:
display(
    takeon_quality_temporal
    .groupby("season")
    .median()
    .reindex(season_order)
)

In [ ]:
persistent_shift_rows = []

for feature in candidate_features:

    temp = (
        season_summary[
            season_summary["feature"] == feature
        ]
        .set_index("season")
    )

    m21 = temp.loc[
        "2021-2022",
        "median"
    ]

    m22 = temp.loc[
        "2022-2023",
        "median"
    ]

    m23 = temp.loc[
        "2023-2024",
        "median"
    ]

    pre_iqr = (
        temp.loc[
            "2021-2022",
            "q75"
        ]
        -
        temp.loc[
            "2021-2022",
            "q25"
        ]
    )

    if pre_iqr > 0:
        shift_22 = (
            (m22 - m21) /
            pre_iqr
        )

        shift_23 = (
            (m23 - m21) /
            pre_iqr
        )
    else:
        shift_22 = np.nan
        shift_23 = np.nan

    persistent_shift_rows.append({
        "feature": feature,
        "2021_22_median": m21,
        "2022_23_median": m22,
        "2023_24_median": m23,
        "shift_2022_23_vs_pre": shift_22,
        "shift_2023_24_vs_pre": shift_23
    })

persistent_shift = pd.DataFrame(
    persistent_shift_rows
)

persistent_shift = persistent_shift.merge(
    feature_registry,
    on="feature",
    how="left"
)

In [ ]:
display(
    persistent_shift.assign(
        persistent_strength=lambda x:
            np.minimum(
                x[
                    "shift_2022_23_vs_pre"
                ].abs(),
                x[
                    "shift_2023_24_vs_pre"
                ].abs()
            )
    )
    .sort_values(
        "persistent_strength",
        ascending=False
    )
    .head(20)
)

## Interpreting Temporal Shifts

Seasonal distribution changes are not automatically treated as data problems.

A genuine football trend may produce gradual movement in features such as:

- crossing;
- shot selection;
- progression;
- positional involvement.

A possible measurement-definition problem is more likely when:

1. a large change occurs abruptly between adjacent seasons;
2. the new level persists afterward;
3. related variables shift simultaneously;
4. the change is consistent with known concerns about the underlying statistic;
5. football interpretation alone does not provide a convincing explanation.

Any eventual adjustment will therefore target specific unstable variables rather than automatically standardizing every feature by season.

## 10B.13 Investigating the 2020-21 Dip

Visual inspection reveals an unexpected pattern across several unrelated attacking features.

A number of measures decline during 2020-21 before recovering in subsequent seasons, including measures of:

- shooting;
- passing progression;
- box delivery;
- carrying progression;
- box penetration.

Because the pattern appears across multiple behavioural families rather than a single statistic, it is less consistent with an isolated feature-definition change.

However, the shift could still arise from:

- genuine season-level football conditions;
- changes in the league composition of the winger sample;
- changes in the types of players classified as wingers;
- differences in which players meet the minutes threshold.

Two additional diagnostics are therefore performed:

1. league-specific seasonal trends;
2. within-player adjacent-season comparisons.

These tests help distinguish a broad season effect from changes in sample composition.

In [ ]:
dip_features = [
    "progressive_carries_p90",
    "carries_into_box_p90",
    "progressive_pass_share",
    "passes_into_box_p90",
    "shots_p90",
    "shots_per_100_att3rd_touches",
    "carry_box_entry_share"
]

In [ ]:
league_season_medians = (
    feature_df
    .groupby(["comp", "season"])[dip_features]
    .median()
    .reset_index()
)

display(league_season_medians.head())

In [ ]:
for feature in dip_features:

    plt.figure(figsize=(10, 5))

    for league in sorted(feature_df["comp"].dropna().unique()):

        temp = (
            league_season_medians[
                league_season_medians["comp"] == league
            ]
            .set_index("season")
            .reindex(season_order)
            .reset_index()
        )

        plt.plot(
            temp["season"],
            temp[feature],
            marker="o",
            label=league
        )

    plt.title(f"{feature} — median by league")
    plt.xlabel("Season")
    plt.ylabel("Median")
    plt.xticks(rotation=45)
    plt.legend()
    plt.show()

In [ ]:
panel_source = feature_df[
    [
        "player_id",
        "player",
        "season",
        "Minutes"
    ] + dip_features
].copy()

In [ ]:
def weighted_mean(group, col):
    values = group[col]
    weights = group["Minutes"]

    valid = values.notna() & weights.notna()

    if valid.sum() == 0:
        return np.nan

    return np.average(
        values[valid],
        weights=weights[valid]
    )

In [ ]:
panel_rows = []

for (player_id, season), group in panel_source.groupby(
    ["player_id", "season"]
):

    row = {
        "player_id": player_id,
        "player": group["player"].iloc[0],
        "season": season,
        "Minutes": group["Minutes"].sum()
    }

    for feature in dip_features:
        row[feature] = weighted_mean(
            group,
            feature
        )

    panel_rows.append(row)

player_season_panel = pd.DataFrame(panel_rows)

print(player_season_panel.shape)

display(player_season_panel.head())

In [ ]:
pre = (
    player_season_panel[
        player_season_panel["season"] == "2019-2020"
    ]
    .set_index("player_id")
)

covid = (
    player_season_panel[
        player_season_panel["season"] == "2020-2021"
    ]
    .set_index("player_id")
)

common_players = pre.index.intersection(
    covid.index
)

print(
    "Players present in both seasons:",
    len(common_players)
)

In [ ]:
paired_changes_1920_2021 = []

for feature in dip_features:

    diff = (
        covid.loc[common_players, feature]
        -
        pre.loc[common_players, feature]
    )

    paired_changes_1920_2021.append({
        "feature": feature,
        "players": diff.notna().sum(),
        "median_change": diff.median(),
        "mean_change": diff.mean(),
        "pct_players_decreasing":
            (diff < 0).mean() * 100
    })

paired_changes_1920_2021 = pd.DataFrame(
    paired_changes_1920_2021
)

display(paired_changes_1920_2021)

In [ ]:
paired_season_rows = []

for i in range(1, len(season_order)):

    previous_season = season_order[i - 1]
    current_season = season_order[i]

    previous = (
        player_season_panel[
            player_season_panel["season"] == previous_season
        ]
        .set_index("player_id")
    )

    current = (
        player_season_panel[
            player_season_panel["season"] == current_season
        ]
        .set_index("player_id")
    )

    common = previous.index.intersection(
        current.index
    )

    for feature in dip_features:

        diff = (
            current.loc[common, feature]
            -
            previous.loc[common, feature]
        )

        paired_season_rows.append({
            "feature": feature,
            "from_season": previous_season,
            "to_season": current_season,
            "common_players": len(common),
            "valid_pairs": diff.notna().sum(),
            "median_change": diff.median(),
            "pct_players_decreasing":
                (diff < 0).mean() * 100
        })

paired_season_changes = pd.DataFrame(
    paired_season_rows
)

In [ ]:
display(
    paired_season_changes[
        paired_season_changes["to_season"]
        == "2020-2021"
    ]
)

In [ ]:
for feature in dip_features:

    temp = paired_season_changes[
        paired_season_changes["feature"] == feature
    ].copy()

    temp["transition"] = (
        temp["from_season"]
        + " → "
        + temp["to_season"]
    )

    plt.figure(figsize=(10, 4))

    plt.bar(
        temp["transition"],
        temp["median_change"]
    )

    plt.axhline(
        0,
        linewidth=1
    )

    plt.title(
        f"Within-player season change — {feature}"
    )

    plt.ylabel("Median within-player change")
    plt.xticks(rotation=45)

    plt.show()

### Interpretation of the 2020-21 Distribution Shift

Several attacking features showed lower aggregate medians during the 2020-21 season.

To determine whether this reflected sample composition or a possible measurement discontinuity, two additional diagnostics were performed:

1. season-level distributions were compared separately across the five leagues;
2. adjacent-season changes were calculated only for players appearing in both seasons.

The decline remained visible across multiple leagues and, for several features, within the same players.

The clearest examples included progressive carrying and shooting propensity. These movements occurred across conceptually different attacking behaviours rather than being isolated to one statistical definition.

The evidence therefore does not support treating the 2020-21 pattern as a simple provider-definition break.

Instead, it is retained as observed season-level football variation.

No correction, exclusion or season-specific normalization is applied.

The within-player comparison is not treated as a perfect causal control because players may change clubs, roles, managers and tactical environments between seasons. It is used only as supporting evidence that the aggregate shift is not explained solely by changes in sample composition.

### Temporal Variation vs Measurement Instability

Three distinct forms of temporal behaviour were observed:

- **broad seasonal shifts**, such as the 2020-21 decline across several attacking behaviours;
- **gradual long-term trends**, such as changes in average shot distance;
- **potential metric-specific discontinuities**, which require separate investigation.

Only the third category would justify statistical correction for historical comparability.

Genuine season-level or long-term football variation is preserved in the dataset.

In [ ]:
takeon_league_season = (
    feature_df
    .groupby(["comp", "season"])
    [
        [
            "take_ons_p90",
            "take_ons_per_100_live_touches"
        ]
    ]
    .median()
    .reset_index()
)

for feature in [
    "take_ons_p90",
    "take_ons_per_100_live_touches"
]:

    plt.figure(figsize=(10, 5))

    for league in sorted(
        feature_df["comp"].dropna().unique()
    ):

        temp = (
            takeon_league_season[
                takeon_league_season["comp"] == league
            ]
            .set_index("season")
            .reindex(season_order)
            .reset_index()
        )

        plt.plot(
            temp["season"],
            temp[feature],
            marker="o",
            label=league
        )

    plt.title(f"{feature} — median by league")
    plt.xlabel("Season")
    plt.ylabel("Median")
    plt.xticks(rotation=45)
    plt.legend()

    plt.show()

# 11. Correlation and Redundancy Audit

The candidate pool intentionally contains multiple representations of related behaviours.

Examples include:

- attacking-third involvement and box involvement;
- progressive carrying and final-third carrying;
- progressive passing and final-third passing;
- action volume and action propensity;
- shooting volume and shooting propensity.

This redundancy was preserved during feature construction so that feature selection would not be based solely on intuition.

At this stage, the objective is to identify variables that provide substantially overlapping information.

Redundancy matters because highly correlated features can unintentionally **overweight a single behavioural dimension** in later distance-based analysis.

For example, if three nearly equivalent carrying features are included while only one receiving feature is included, player similarity may become disproportionately driven by carrying behaviour.

Correlation is therefore used as a diagnostic for redundancy rather than as an automatic feature-removal rule.

Feature decisions combine:

1. statistical similarity;
2. football interpretation;
3. conceptual uniqueness;
4. robustness and data quality.

## 11.1 Correlation Strategy

Two correlation measures are examined.

### Spearman correlation

Spearman correlation evaluates whether two features move monotonically together without requiring a linear relationship.

It is used as the primary redundancy diagnostic because many football features are skewed or bounded.

### Pearson correlation

Pearson correlation is retained as a secondary diagnostic for strong linear relationships.

A feature pair that is highly correlated under both measures provides stronger evidence of redundancy than a pair flagged by only one.

No correlation threshold automatically determines feature removal.

In [ ]:
candidate_matrix = feature_df[candidate_features].copy()

spearman_corr = candidate_matrix.corr(
    method="spearman"
)

pearson_corr = candidate_matrix.corr(
    method="pearson"
)

print("Spearman matrix shape:", spearman_corr.shape)
print("Pearson matrix shape:", pearson_corr.shape)

In [ ]:
correlation_pairs = []

for i, feature_1 in enumerate(candidate_features):

    for j in range(i + 1, len(candidate_features)):

        feature_2 = candidate_features[j]

        correlation_pairs.append({
            "feature_1": feature_1,
            "feature_2": feature_2,
            "spearman": spearman_corr.loc[
                feature_1,
                feature_2
            ],
            "pearson": pearson_corr.loc[
                feature_1,
                feature_2
            ]
        })

correlation_pairs = pd.DataFrame(
    correlation_pairs
)

correlation_pairs["abs_spearman"] = (
    correlation_pairs["spearman"].abs()
)

correlation_pairs["abs_pearson"] = (
    correlation_pairs["pearson"].abs()
)

correlation_pairs = correlation_pairs.sort_values(
    "abs_spearman",
    ascending=False
)

display(
    correlation_pairs.head(40)
)

In [ ]:
registry_lookup = (
    feature_registry
    .set_index("feature")
)

correlation_pairs["family_1"] = (
    correlation_pairs["feature_1"]
    .map(registry_lookup["family"])
)

correlation_pairs["family_2"] = (
    correlation_pairs["feature_2"]
    .map(registry_lookup["family"])
)

correlation_pairs["status_1"] = (
    correlation_pairs["feature_1"]
    .map(registry_lookup["status"])
)

correlation_pairs["status_2"] = (
    correlation_pairs["feature_2"]
    .map(registry_lookup["status"])
)

In [ ]:
display(
    correlation_pairs[
        [
            "feature_1",
            "family_1",
            "status_1",
            "feature_2",
            "family_2",
            "status_2",
            "spearman",
            "pearson"
        ]
    ]
    .head(40)
)

In [ ]:
strong_pairs = correlation_pairs[
    correlation_pairs["abs_spearman"] >= 0.80
].copy()

display(
    strong_pairs[
        [
            "feature_1",
            "family_1",
            "feature_2",
            "family_2",
            "spearman",
            "pearson"
        ]
    ]
)

In [ ]:
moderate_high_pairs = correlation_pairs[
    (
        correlation_pairs["abs_spearman"] >= 0.65
    )
    &
    (
        correlation_pairs["abs_spearman"] < 0.80
    )
].copy()

display(
    moderate_high_pairs[
        [
            "feature_1",
            "family_1",
            "feature_2",
            "family_2",
            "spearman",
            "pearson"
        ]
    ]
    .head(40)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(16, 14)
)

im = ax.imshow(
    spearman_corr,
    vmin=-1,
    vmax=1,
    aspect="auto",
    cmap = "Spectral"
)

ax.set_xticks(
    range(len(candidate_features))
)

ax.set_yticks(
    range(len(candidate_features))
)

ax.set_xticklabels(
    candidate_features,
    rotation=90
)

ax.set_yticklabels(
    candidate_features
)

ax.set_title(
    "Candidate Feature Spearman Correlation"
)

fig.colorbar(
    im,
    ax=ax,
    label="Spearman correlation"
)

plt.tight_layout()
plt.show()

### 11.4.1 Feature Selection and Pruning

The initial correlation analysis identifies statistical overlap among the
27 engineered candidate features.

Feature selection will be conducted in two stages:

**Stage 1 — Structural and conceptual screening**

Remove variables that are mathematically redundant, insufficiently
distinct, or more appropriate as supporting/contextual descriptors.

**Stage 2 — Empirical redundancy assessment**

Investigate strongly correlated features that survive the initial
screening.

The following questions guide the assessment:

1. Do the variables measure the same underlying football behaviour?
2. Do they distinguish meaningfully different player profiles despite
   their overall correlation?
3. Does removing a variable substantially affect stylistic similarity?
4. Would retaining both variables unintentionally overweight one
   behavioural dimension?

The final representation is selected through football interpretation
and empirical evidence, rather than an automatic correlation threshold.

In [ ]:
initial_exclusions = {
    "box_touch_share": (
        "Drop",
        "Mathematically determined by attacking_third_touch_share "
        "and box_penetration_touch_share."
    ),

    "carries_into_final_third_p90": (
        "Drop",
        "Overlaps with progressive carrying and box-carrying measures; "
        "limited additional behavioural distinction."
    ),

    "passes_into_final_third_p90": (
        "Drop",
        "Overlaps with progressive passing and penalty-area passing; "
        "limited additional behavioural distinction."
    ),

    "passes_attempted_p90": (
        "Move to context",
        "General passing involvement and possession context rather "
        "than a sufficiently distinctive attacking-style variable."
    ),

    "key_passes_p90": (
        "Move to support",
        "Depends on a subsequent shot and partly reflects "
        "downstream attacking outcomes."
    ),

    "npxg_per_shot": (
        "Move to support",
        "Reflects shot location and opportunity quality; "
        "avg_shot_distance provides a cleaner location tendency."
    )
}

provisional_features = [
    feature
    for feature in candidate_features
    if feature not in initial_exclusions
]

print("Original candidates:", len(candidate_features))
print("Initial exclusions:", len(initial_exclusions))
print("Provisional features:", len(provisional_features))

assert len(provisional_features) == 21

remaining_strong_pairs = strong_pairs[
    strong_pairs["feature_1"].isin(provisional_features)
    &
    strong_pairs["feature_2"].isin(provisional_features)
].copy()

print(
    "\nStrongly correlated pairs remaining:",
    len(remaining_strong_pairs)
)

display(
    remaining_strong_pairs[
        [
            "feature_1",
            "feature_2",
            "spearman",
            "pearson"
        ]
    ]
)

### 11.4.2 Behavioural Distinctiveness of Correlated Features

A high correlation does not necessarily imply that two features
describe interchangeable behaviours.

For example:

- Crosses per 90 represents actual match-level crossing volume.
- Crosses as a share of attempted passes represents crossing
  propensity relative to passing involvement.

A player can rank relatively low on one and high on the other.

To investigate such differences, percentile rankings are compared
for each strongly correlated feature pair.

Large differences identify player spells for which the two variables
provide contrasting behavioural descriptions.

A 25-percentile-point difference is used as a descriptive inspection
threshold, not as an automatic feature-retention rule.

In [ ]:
# Percentile ranks among the provisional features
feature_ranks = feature_df[
    provisional_features
].rank(pct=True)

rank_audit = []
rank_examples = []

for _, pair in remaining_strong_pairs.iterrows():

    a = pair["feature_1"]
    b = pair["feature_2"]

    gap = feature_ranks[a] - feature_ranks[b]

    rank_audit.append({
        "feature_1": a,
        "feature_2": b,
        "spearman": pair["spearman"],
        "mean_gap_pp": gap.abs().mean() * 100,
        "pct_gap_ge_25pp": (
            (gap.abs() >= 0.25).mean() * 100
        )
    })

    # Largest disagreement in each direction
    for idx in [gap.idxmax(), gap.idxmin()]:

        rank_examples.append({
            "player": feature_df.loc[idx, "player"],
            "squad": feature_df.loc[idx, "squad"],
            "season": feature_df.loc[idx, "season"],
            "feature_1": a,
            "feature_2": b,
            "feature_1_percentile": (
                feature_ranks.loc[idx, a] * 100
            ),
            "feature_2_percentile": (
                feature_ranks.loc[idx, b] * 100
            ),
            "rank_gap_pp": gap.loc[idx] * 100
        })

rank_audit = pd.DataFrame(rank_audit)
rank_examples = pd.DataFrame(rank_examples)

display(rank_audit.round(2))

print("\nPlayer examples with contrasting rankings:")
display(rank_examples.round(1))

### 11.4.3 Nearest-Neighbour Sensitivity

The second redundancy diagnostic examines whether removing a
correlated feature materially changes player similarity.

A temporary standardized representation of the 21 provisional
features is constructed.

For each feature involved in a strong-correlation pair:

1. Identify every observation's ten nearest neighbours using
   Euclidean distance.
2. Remove the feature and recompute the neighbours.
3. Measure the proportion of neighbours that remain unchanged.

This is a local sensitivity diagnostic rather than the project's
final player-similarity model.

The purpose is to understand the influence of individual features
on the geometry of the representation.

Neither high neighbour overlap nor low overlap independently
determines whether a feature should be retained.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

# Temporary scaling for diagnostic purposes only
temp_scaler = StandardScaler()

temp_X = pd.DataFrame(
    temp_scaler.fit_transform(
        feature_df[provisional_features]
    ),
    columns=provisional_features,
    index=feature_df.index
)

def nearest_neighbours(X, k=10):

    distances = pairwise_distances(
        X,
        metric="euclidean"
    )

    np.fill_diagonal(distances, np.inf)

    return np.argpartition(
        distances,
        kth=k - 1,
        axis=1
    )[:, :k]


k = 10

baseline_neighbours = nearest_neighbours(
    temp_X,
    k=k
)

# Only features involved in remaining strong pairs
review_features = sorted(
    set(remaining_strong_pairs["feature_1"])
    |
    set(remaining_strong_pairs["feature_2"])
)

sensitivity_results = []

for feature in review_features:

    alternative_X = temp_X.drop(
        columns=[feature]
    )

    alternative_neighbours = nearest_neighbours(
        alternative_X,
        k=k
    )

    overlaps = np.array([
        len(set(original) & set(alternative)) / k
        for original, alternative in zip(
            baseline_neighbours,
            alternative_neighbours
        )
    ])

    sensitivity_results.append({
        "removed_feature": feature,
        "mean_top10_overlap": overlaps.mean(),
        "mean_neighbours_changed": (
            k * (1 - overlaps.mean())
        ),
        "pct_3plus_changed": (
            (overlaps <= 0.70).mean() * 100
        )
    })

sensitivity_results = (
    pd.DataFrame(sensitivity_results)
    .sort_values("mean_top10_overlap")
)

display(sensitivity_results.round(3))

### 11.4.4 Final Feature-Selection Decisions

The redundancy review provides three main conclusions.

**Crossing and take-on volume versus propensity**

Although these feature pairs are strongly correlated, their
denominators represent different football concepts. Their
player-ranking disagreements support retaining both
representations.

**Carrying propensity versus progression method**

Progressive carry share describes the proportion of carries that
are progressive. Carry progression share instead describes the
relative use of carrying versus passing for progression.

These are related but distinct behavioural questions.

**Progressive distance per carry**

This variable has substantial statistical overlap with multiple
carrying measures.

Although it identifies some distinctive player profiles, its
incremental contribution must be weighed against the risk of
overrepresenting the carrying dimension.

The final selection is recorded explicitly, including the reason
for every exclusion or retained high-correlation feature.

In [ ]:
# Additional decisions following redundancy diagnostics
additional_exclusions = {
    "progressive_distance_per_carry": (
        "Drop",
        "Strong overlap with progressive_carry_share and "
        "progressive_carries_p90; excluded to limit "
        "overweighting of carrying behaviour."
    )
}

# Combine all exclusions
all_exclusions = {
    **initial_exclusions,
    **additional_exclusions
}

# Create final decision registry
feature_decisions = feature_registry.copy()

feature_decisions["decision"] = "Retain"
feature_decisions["reason"] = ""

for feature, (decision, reason) in all_exclusions.items():

    mask = feature_decisions["feature"] == feature

    assert mask.sum() == 1, feature

    feature_decisions.loc[mask, "decision"] = decision
    feature_decisions.loc[mask, "reason"] = reason


# Document deliberately retained correlated variables
retention_reasons = {
    "crosses_p90":
        "Retained as crossing volume per 90; distinct from "
        "crossing propensity relative to passing.",

    "cross_pass_share":
        "Retained as crossing propensity; distinguishes crossing "
        "orientation from total crossing volume.",

    "take_ons_p90":
        "Retained as take-on volume per 90; distinct from "
        "per-touch dribbling propensity.",

    "take_ons_per_100_live_touches":
        "Retained as take-on propensity relative to involvement; "
        "captures behaviour not fully reflected by volume.",

    "progressive_carry_share":
        "Retained as progressive carrying propensity among "
        "all carries.",

    "carry_progression_share":
        "Retained as relative choice between carrying and "
        "passing for progression.",

    "long_pass_share":
        "Retained as passing-profile orientation; monitored "
        "for overlap with crossing."
}

for feature, reason in retention_reasons.items():

    mask = (
        (feature_decisions["feature"] == feature)
        &
        (feature_decisions["decision"] == "Retain")
    )

    feature_decisions.loc[mask, "reason"] = reason


retained_candidate_features = (
    feature_decisions.loc[
        feature_decisions["decision"] == "Retain",
        "feature"
    ].tolist()
)

# Move reclassified features to supporting descriptors
support_additions = (
    feature_decisions.loc[
        feature_decisions["decision"] == "Move to support",
        "feature"
    ].tolist()
)

for feature in support_additions:

    support_df[feature] = feature_df[feature]

    if feature not in support_features:
        support_features.append(feature)


# Preserve context features separately
context_features = (
    feature_decisions.loc[
        feature_decisions["decision"] == "Move to context",
        "feature"
    ].tolist()
)

context_df = feature_df[
    identity_cols + context_features
].copy()


# Final validation
assert set(retained_candidate_features).isdisjoint(
    all_exclusions
)

assert (
    feature_decisions["feature"].nunique()
    == len(candidate_features)
)

assert (
    len(retained_candidate_features)
    + len(all_exclusions)
    == len(candidate_features)
)

print("Original candidates:", len(candidate_features))
print("Removed/reclassified:", len(all_exclusions))
print("Final style features:", len(retained_candidate_features))
print("Supporting descriptors:", len(support_features))

display(
    feature_decisions[
        ["feature", "family", "status", "decision", "reason"]
    ]
)

print("\nFinal retained features:")
for feature in retained_candidate_features:
    print("-", feature)

In [ ]:
final_strong_pairs = correlation_pairs[
    correlation_pairs["feature_1"].isin(
        retained_candidate_features
    )
    &
    correlation_pairs["feature_2"].isin(
        retained_candidate_features
    )
    &
    (correlation_pairs["abs_spearman"] >= 0.80)
].copy()

print(
    "Remaining strong-correlation pairs:",
    len(final_strong_pairs)
)

display(
    final_strong_pairs[
        [
            "feature_1",
            "feature_2",
            "spearman",
            "pearson"
        ]
    ]
)

## 11.5 Structural Redundancy

Some candidate features are related not only statistically but mathematically.

For example:

$$
BoxTouchShare =
\frac{AttackingPenaltyAreaTouches}{TotalTouches}
$$

can be decomposed as:

$$
\frac{AttackingThirdTouches}{TotalTouches}
\times
\frac{AttackingPenaltyAreaTouches}{AttackingThirdTouches}
$$

Therefore:

`box_touch_share`

contains no completely independent information if both:

- `attacking_third_touch_share`
- `box_penetration_touch_share`

are already retained.

This type of structural redundancy is treated more strongly than ordinary statistical correlation.

In [ ]:
abs_spearman = spearman_corr.abs()

mean_abs_corr = (
    (
        abs_spearman.sum(axis=1) - 1
    )
    /
    (
        len(candidate_features) - 1
    )
)

max_abs_corr = (
    abs_spearman
    .where(
        ~np.eye(
            len(abs_spearman),
            dtype=bool
        )
    )
    .max(axis=1)
)

correlation_centrality = pd.DataFrame({
    "feature": candidate_features,
    "mean_abs_correlation": [
        mean_abs_corr[f]
        for f in candidate_features
    ],
    "max_abs_correlation": [
        max_abs_corr[f]
        for f in candidate_features
    ]
})

correlation_centrality = (
    correlation_centrality
    .merge(
        feature_registry,
        on="feature",
        how="left"
    )
    .sort_values(
        "mean_abs_correlation",
        ascending=False
    )
)

display(correlation_centrality)

### Interpreting Cross-Family Correlations

Correlation between conceptually different football behaviours is expected.

For example, advanced players may simultaneously:

- receive more progressive passes;
- take more shots;
- enter the penalty area more frequently.

Such relationships reflect football structure and do not automatically imply that one variable should be removed.

Feature removal is prioritized when:

1. the variables measure essentially the same behaviour;
2. one variable is mathematically derived from another;
3. correlation is extremely high;
4. one representation has a clearer football interpretation or stronger data quality.

# 12. Final Matrix Design and Scaling

After conceptual screening, feature-quality auditing and the first redundancy pass, the candidate playstyle representation has been reduced from 27 to 20 behavioural features.

The earlier expectation of approximately 12–18 final dimensions was a planning guideline rather than a fixed target.

Further features are not removed solely to satisfy a predetermined feature count. A variable is retained when it represents a conceptually distinct behaviour and has passed the preceding quality checks.

Before modelling, two additional issues must be addressed:

1. **feature-family balance** — ensuring that one behavioural family does not dominate simply because it contains more variables;
2. **feature scaling** — putting features measured in different units onto a comparable numerical scale.

The primary modelling matrix will use standardized features with equal feature-level weighting.

A family-balanced version will also be constructed later as a sensitivity analysis.

In [ ]:
macro_family_map = {
    # Territory
    "attacking_third_touch_share": "Territory",
    "box_penetration_touch_share": "Territory",

    # Receiving
    "progressive_receptions_p90": "Receiving",
    "progressive_reception_share": "Receiving",

    # Carrying
    "progressive_carries_p90": "Carrying",
    "progressive_carry_share": "Carrying",
    "carries_into_box_p90": "Carrying",
    "progressive_distance_per_carry": "Carrying",

    # Dribbling
    "take_ons_p90": "Dribbling",
    "take_ons_per_100_live_touches": "Dribbling",

    # Passing
    "progressive_passes_p90": "Passing",
    "progressive_pass_share": "Passing",
    "passes_into_box_p90": "Passing",
    "long_pass_share": "Passing",

    # Crossing
    "crosses_p90": "Crossing",
    "cross_pass_share": "Crossing",

    # Shooting
    "shots_p90": "Shooting",
    "shots_per_100_att3rd_touches": "Shooting",
    "avg_shot_distance": "Shooting",

    # Relative behavioural choice
    "carry_progression_share": "Behavioural Choice",
    "carry_box_entry_share": "Behavioural Choice"
}

In [ ]:
macro_family_map = {
    feature: family
    for feature, family in macro_family_map.items()
    if feature in retained_candidate_features
}

assert set(macro_family_map) == set(retained_candidate_features)

In [ ]:
assert set(macro_family_map.keys()) == set(retained_candidate_features)

print("All retained features assigned to a macro family.")

In [ ]:
family_balance = (
    pd.Series(macro_family_map)
    .value_counts()
    .rename_axis("family")
    .reset_index(name="n_features")
)

family_balance["share_of_features_pct"] = (
    family_balance["n_features"] /
    len(retained_candidate_features) *
    100
)

display(family_balance)

## 12.1 Feature-Family Balance

The retained candidate set contains eight broader behavioural families.

The largest family, Passing, contain four features, while other families contain two or three.

This is considered sufficiently balanced to avoid additional feature deletion solely on the basis of feature count.

Importantly, the retained variables within the larger families describe distinct concepts such as:


- volume;
- orientation;
- territorial penetration;
- action characteristics.

Further pruning at this stage would risk removing meaningful behavioural distinctions simply to achieve an arbitrary feature total.

The 20-feature representation is therefore retained for the initial modelling stage.

In [ ]:
style_matrix_raw = (
    feature_df[
        retained_candidate_features
    ]
    .copy()
)

retained_missing = (
    style_matrix_raw
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_n")
)

retained_missing["missing_pct"] = (
    retained_missing["missing_n"] /
    len(style_matrix_raw) *
    100
)

display(retained_missing)

In [ ]:
print(
    "Total missing cells:",
    style_matrix_raw.isna().sum().sum()
)

In [ ]:
finite_check = np.isfinite(
    style_matrix_raw.to_numpy(dtype=float)
)

print(
    "Non-finite values:",
    (~finite_check).sum()
)

In [ ]:
assert style_matrix_raw.isna().sum().sum() == 0

assert np.isfinite(
    style_matrix_raw.to_numpy(dtype=float)
).all()

## 12.2 Feature Scaling

The retained features use different numerical units.

Examples include:

- proportions;
- actions per 90;
- actions per 100 touches;
- distances.

Using the raw values directly in PCA, clustering or distance-based similarity would therefore cause features with larger numerical scales to dominate the analysis.

The primary feature matrix is standardized using:

$$
z =
\frac{x-\mu}{\sigma}
$$

Each feature therefore has approximately:

- mean = 0;
- standard deviation = 1.

Standardization is performed across the complete player-season population rather than independently by season.

Season-specific standardization is deliberately avoided because the temporal audit identified genuine historical football variation that should remain visible in the data.

Standardization changes numerical scale, not the underlying ranking or interpretation of a feature.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

style_matrix_scaled = pd.DataFrame(
    scaler.fit_transform(style_matrix_raw),
    columns=retained_candidate_features,
    index=feature_df.index
)

In [ ]:
scaling_check = pd.DataFrame({
    "mean": style_matrix_scaled.mean(),
    "std_population": style_matrix_scaled.std(ddof=0)
})

display(scaling_check)

### Distribution Shape and Transformation

No logarithmic, percentile or quantile transformation is applied automatically to skewed features.

Several football behaviours are naturally concentrated among relatively small groups of players.

For example, unusually high dribbling, carrying or shooting volume may represent genuine stylistic extremity rather than statistical noise.

The preceding feature audit found no justification for suppressing such observations solely to make the distributions more Gaussian.

The primary modelling matrix therefore preserves the original engineered feature distributions and applies only standardization.

Alternative robust scaling may later be used as a sensitivity test.

In [ ]:
style_matrix = pd.concat(
    [
        feature_df[identity_cols].reset_index(drop=True),
        style_matrix_scaled.reset_index(drop=True)
    ],
    axis=1
)

print(style_matrix.shape)

display(style_matrix.head())

In [ ]:
style_matrix_unscaled = pd.concat(
    [
        feature_df[identity_cols].reset_index(drop=True),
        style_matrix_raw.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
family_counts = (
    pd.Series(macro_family_map)
    .value_counts()
)

In [ ]:
family_weights = {
    feature: 1 / np.sqrt(
        family_counts[
            macro_family_map[feature]
        ]
    )
    for feature in retained_candidate_features
}

family_weights

In [ ]:
style_matrix_family_balanced = (
    style_matrix_scaled.copy()
)

for feature in retained_candidate_features:

    style_matrix_family_balanced[feature] *= (
        family_weights[feature]
    )

## 12.3 Family-Balanced Sensitivity Representation

Standardized features receive equal feature-level weight in the primary model.

Because behavioural families contain different numbers of features, an alternative family-balanced matrix is also constructed.

For a feature belonging to a family containing \(n\) variables:

$$
weight =
\frac{1}{\sqrt{n}}
$$

This approximately equalizes the total potential contribution of each behavioural family.

The family-balanced representation is **not treated as the primary model**.

Instead, it will later be used as a sensitivity analysis:

> Do player similarities and discovered archetypes remain broadly stable when behavioural families are given more equal aggregate weight?

If results are stable across both representations, confidence in the discovered playstyle structure increases.

In [ ]:
final_feature_spec = (
    feature_decisions[
        feature_decisions["decision"] == "Retain"
    ][
        [
            "feature",
            "family",
            "status",
            "decision",
            "reason"
        ]
    ]
    .copy()
)

final_feature_spec["macro_family"] = (
    final_feature_spec["feature"]
    .map(macro_family_map)
)

final_feature_spec["family_weight"] = (
    final_feature_spec["feature"]
    .map(family_weights)
)

display(final_feature_spec)

# Final Candidate Style Representation

Following conceptual design, feature construction, quality auditing, temporal analysis and redundancy review, the playstyle representation contains **20 retained behavioural features**.

These cover eight broader dimensions:

1. Territory
2. Receiving
3. Carrying
4. Dribbling
5. Passing
6. Crossing
7. Shooting
8. Relative behavioural choice

All retained variables have:

- a defined football interpretation;
- passed structural sanity checks;
- been reviewed for temporal comparability;
- survived the first redundancy analysis.

The modelling pipeline will use the globally standardized 20-feature matrix as its primary representation.

A family-balanced version is retained for later robustness testing.

Quality, output and contextual variables remain outside the playstyle feature space and will be reintroduced only after style discovery for interpretation and player evaluation.

# 13. Final Matrix Quality Assurance

Before exporting the engineered playstyle data, the final matrices are checked for:

- preservation of the canonical player × club × season population;
- duplicate observations;
- missing and non-finite modelling values;
- expected feature count;
- correct standardization;
- alignment between raw, standardized and family-balanced representations.

This final check is intended to ensure that feature engineering has changed the representation of the data without changing the underlying analytical population.

In [ ]:
print("Canonical enriched dataset:", df.shape)
print("Engineered feature table:", feature_df.shape)
print("Unscaled style matrix:", style_matrix_unscaled.shape)
print("Scaled style matrix:", style_matrix.shape)
print(
    "Family-balanced matrix:",
    style_matrix_family_balanced.shape
)
print("Supporting descriptor table:", support_df.shape)

print(
    "\nRetained behavioural features:",
    len(retained_candidate_features)
)

In [ ]:
qa_key = [
    "player_id",
    "squad",
    "season"
]

for name, dataframe in {
    "feature_df": feature_df,
    "style_matrix_unscaled": style_matrix_unscaled,
    "style_matrix": style_matrix,
    "support_df": support_df
}.items():

    duplicates = dataframe.duplicated(
        subset=qa_key
    ).sum()

    print(
        f"{name}: "
        f"{len(dataframe)} rows, "
        f"{duplicates} duplicate canonical keys"
    )

In [ ]:
assert len(feature_df) == len(df) == 1396
assert len(style_matrix_unscaled) == 1396
assert len(style_matrix) == 1396
assert len(support_df) == 1396

assert not feature_df.duplicated(
    subset=qa_key
).any()

assert not style_matrix_unscaled.duplicated(
    subset=qa_key
).any()

assert not style_matrix.duplicated(
    subset=qa_key
).any()

assert not support_df.duplicated(
    subset=qa_key
).any()

print("Population and grain checks passed.")

In [ ]:
alignment_cols = [
    "player_id",
    "squad",
    "season"
]

assert (
    style_matrix_unscaled[alignment_cols]
    .reset_index(drop=True)
    .equals(
        style_matrix[alignment_cols]
        .reset_index(drop=True)
    )
)

assert (
    style_matrix[alignment_cols]
    .reset_index(drop=True)
    .equals(
        support_df[alignment_cols]
        .reset_index(drop=True)
    )
)

print("Row alignment across analytical tables passed.")

In [ ]:
final_raw_matrix = style_matrix_unscaled[
    retained_candidate_features
]

final_scaled_matrix = style_matrix[
    retained_candidate_features
]

print(
    "Raw missing values:",
    final_raw_matrix.isna().sum().sum()
)

print(
    "Scaled missing values:",
    final_scaled_matrix.isna().sum().sum()
)

print(
    "Raw non-finite values:",
    (
        ~np.isfinite(
            final_raw_matrix.to_numpy(dtype=float)
        )
    ).sum()
)

print(
    "Scaled non-finite values:",
    (
        ~np.isfinite(
            final_scaled_matrix.to_numpy(dtype=float)
        )
    ).sum()
)

In [ ]:
assert final_raw_matrix.isna().sum().sum() == 0
assert final_scaled_matrix.isna().sum().sum() == 0

assert np.isfinite(
    final_raw_matrix.to_numpy(dtype=float)
).all()

assert np.isfinite(
    final_scaled_matrix.to_numpy(dtype=float)
).all()

print("Final modelling matrices contain no missing or non-finite values.")

In [ ]:
final_scaling_check = pd.DataFrame({
    "mean": final_scaled_matrix.mean(),
    "std": final_scaled_matrix.std(ddof=0)
})

final_scaling_check["abs_mean"] = (
    final_scaling_check["mean"].abs()
)

final_scaling_check["std_error"] = (
    final_scaling_check["std"] - 1
).abs()

display(final_scaling_check)

In [ ]:
assert (
    final_scaling_check["abs_mean"]
    < 1e-10
).all()

assert (
    final_scaling_check["std_error"]
    < 1e-10
).all()

print("Standardization check passed.")

In [ ]:
assert list(
    style_matrix_family_balanced.columns
) == retained_candidate_features

assert len(
    style_matrix_family_balanced
) == len(df)

assert np.isfinite(
    style_matrix_family_balanced
    .to_numpy(dtype=float)
).all()

print(
    "Family-balanced matrix check passed."
)

In [ ]:
style_matrix_family_balanced_export = pd.concat(
    [
        feature_df[
            identity_cols
        ].reset_index(drop=True),

        style_matrix_family_balanced
        .reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
final_inventory = pd.DataFrame({
    "feature": retained_candidate_features
})

final_inventory["macro_family"] = (
    final_inventory["feature"]
    .map(macro_family_map)
)

final_inventory["original_status"] = (
    final_inventory["feature"]
    .map(
        feature_registry
        .set_index("feature")["status"]
    )
)

final_inventory["family_weight"] = (
    final_inventory["feature"]
    .map(family_weights)
)

display(final_inventory)

In [ ]:
display(
    final_inventory["macro_family"]
    .value_counts()
    .rename("n_features")
    .to_frame()
)

# 14. Export Analytical Outputs

Notebook 04 produces several outputs rather than a single dataset because different stages of the later analysis require different representations.

### Raw engineered style matrix

Contains the 20 retained behavioural features in their original interpretable units.

Used for:

- player-profile interpretation;
- reporting;
- visualizations;
- football-level explanation.

### Standardized style matrix

Contains the same 21 features standardized across the complete player-season population.

This is the primary input for:

- PCA;
- similarity analysis;
- clustering;
- other distance-based modelling.

### Family-balanced standardized matrix

Contains the standardized features with additional family-level weighting.

This is retained as a sensitivity-analysis representation rather than the primary model.

### Supporting descriptors

Contains quality, output and supplemental role variables deliberately excluded from style discovery.

These will later be used to evaluate and interpret the discovered playstyles.

### Feature specification

Records the final retained feature set, conceptual family and family-balancing weight.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/decoding_winger_playstyles"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(OUTPUT_DIR)

In [ ]:
style_matrix_unscaled.to_csv(
    OUTPUT_DIR /
    "winger_style_features_raw.csv",
    index=False
)

style_matrix_unscaled.to_parquet(
    OUTPUT_DIR /
    "winger_style_features_raw.parquet",
    index=False
)

In [ ]:
style_matrix.to_csv(
    OUTPUT_DIR /
    "winger_style_features_scaled.csv",
    index=False
)

style_matrix.to_parquet(
    OUTPUT_DIR /
    "winger_style_features_scaled.parquet",
    index=False
)

In [ ]:
style_matrix_family_balanced_export.to_csv(
    OUTPUT_DIR /
    "winger_style_features_family_balanced.csv",
    index=False
)

style_matrix_family_balanced_export.to_parquet(
    OUTPUT_DIR /
    "winger_style_features_family_balanced.parquet",
    index=False
)

In [ ]:
support_df.to_csv(
    OUTPUT_DIR /
    "winger_style_supporting_descriptors.csv",
    index=False
)

support_df.to_parquet(
    OUTPUT_DIR /
    "winger_style_supporting_descriptors.parquet",
    index=False
)

In [ ]:
feature_spec_export = final_feature_spec[
    [
        "feature",
        "macro_family",
        "family",
        "status",
        "family_weight",
        "reason"
    ]
].copy()

feature_spec_export.to_csv(
    OUTPUT_DIR /
    "winger_style_feature_specification.csv",
    index=False
)

display(feature_spec_export)

In [ ]:
feature_decisions.to_csv(
    OUTPUT_DIR /
    "winger_style_feature_decisions.csv",
    index=False
)

In [ ]:
exported_files = sorted(
    OUTPUT_DIR.iterdir()
)

for path in exported_files:
    print(
        path.name,
        "-",
        round(
            path.stat().st_size / 1024,
            1
        ),
        "KB"
    )

In [ ]:
test_scaled = pd.read_parquet(
    OUTPUT_DIR /
    "winger_style_features_scaled.parquet"
)

print(test_scaled.shape)

assert test_scaled.shape == style_matrix.shape

assert list(
    test_scaled.columns
) == list(
    style_matrix.columns
)

print("Export round-trip check passed.")

# 15. Conclusion

This notebook converted the enriched winger dataset into a modelling-ready representation of attacking playstyle.

The process began with 27 conceptually motivated candidate features covering:

- territory;
- receiving;
- carrying;
- dribbling;
- passing;
- crossing;
- shooting;
- relative attacking choices.

Candidate variables were evaluated through:

- definition-level sanity checks;
- extreme-player inspection;
- missingness and sparsity analysis;
- denominator reliability checks;
- temporal-stability analysis;
- within-player seasonal comparisons;
- league-level temporal comparisons;
- correlation and redundancy analysis.

Initial structural and conceptual screening removed or reclassified six of the 27 candidates, leaving 21 provisional features. The subsequent redundancy review excluded `progressive_distance_per_carry` to limit overweighting of carrying behaviour, leaving 20 final modelling features. This preserves the documented progression: **27 candidates → 21 provisional features → 20 final features**.

The resulting attacking-style representation contains **20 behavioural features across eight macro families**.

Two modelling representations are retained:

1. a globally standardized feature matrix as the primary representation;
2. a family-balanced standardized matrix for later sensitivity testing.

Quality and output metrics remain deliberately separate from the style matrix.

This completes feature engineering.

The next stage will examine the underlying structure of the 20-dimensional playstyle space before any player archetypes are imposed.

In [ ]:
import joblib

joblib.dump(
    scaler,
    OUTPUT_DIR /
    "style_standard_scaler.joblib"
)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('decoding_winger_playstyles', 'zip', '/content/decoding_winger_playstyles')

files.download('decoding_winger_playstyles.zip')
